# Computação Gráfica e Realidade Aumentada 
## 2025
<author>Authors: Guilherme Cabaço e Henrique Gomes</author>
<h1>Assignment 3 - AR</h1>

In [ ]:
%%html
<!--- 
The camera widget must be visible on some devices for the video capture to work.
--->

<video id="video" autoplay style="width:640px; height:480px;"></video> 
<!---
<video id="videofile" controls autoplay loop style="width:640px; height:480px;" crossOrigin="Anonymous"; src="https://is3l.isr.uc.pt/~pm/CGRA/marcador65.mp4"></video>
 --->

<canvas id="myCanvas" width="640" height="480" style="border:2px solid #000000;">
      Error: Your browser does not support the HTML canvas tag.
</canvas>

In [ ]:
%%html


<!--- all the access to the auxiliary files must use https instead of http --->
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecshader.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecapp.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/cgraobject.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/cgratexture.js"></script>
<script src='https://git.io/glm-js.min.js'></script>

<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/js-aruco/cv.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/js-aruco/aruco.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/js-aruco/svd.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/js-aruco/posit1.js"></script>

<script id="my-vertex-shader" type="x-shader/x-vertex">
precision mediump float;

attribute  vec3 in_Position;
attribute  vec3 in_Color;
uniform mat4 MVP;

varying  vec3 ex_Color;

void main() {
  
    gl_Position = MVP * vec4(in_Position.x, in_Position.y, in_Position.z, 1.0);

    ex_Color = in_Color;
}
</script>

<script id="my-fragment-shader" type="x-shader/x-fragment">
precision mediump float;

varying  vec3 ex_Color;

void main() {
  
    gl_FragColor = vec4(ex_Color,1.0);
}
</script>


<script id="my-vertex-shaderC" type="x-shader/x-vertex">
precision mediump float;

attribute  vec3 in_Position;
attribute  vec3 in_Color;
uniform mat4 MVP;
uniform vec3 un_Color;
varying  vec3 ex_Color;

void main(void) {
  
    gl_Position = MVP * vec4(in_Position.x, in_Position.y, in_Position.z, 1.0);

    ex_Color = un_Color;
}
</script>


<script id="my-vertex-shaderT" type="x-shader/x-vertex">
precision mediump float;

attribute  vec3 in_Position;
attribute  vec3 in_Color;
attribute vec2 in_Tex_Coord;
uniform mat4 MVP;

varying  vec3 ex_Color;
varying  vec2 vTextureCoord;

void main() {
  
    gl_Position = MVP * vec4(in_Position.x, in_Position.y, in_Position.z, 1.0);
    vTextureCoord = in_Tex_Coord;
    ex_Color = in_Color;
}
</script>

<script id="my-fragment-shaderT" type="x-shader/x-fragment">
precision mediump float;
varying  vec2 vTextureCoord;
varying  vec3 ex_Color;

uniform sampler2D uSampler;

void main() {
     gl_FragColor = texture2D(uSampler, vTextureCoord);
}
</script>




### Humanoid shaders

In [ ]:
%%html
<script id="bones-vertex-shader" type="x-shader/x-vertex">
precision mediump float;

uniform mat4 MVP;

attribute vec3 vVertex;
attribute vec2 vTexCoords;
attribute vec3 vNormal;

attribute vec3 in_Color;      
varying vec3 ex_Color;        

attribute vec4 BoneIDs;
attribute vec4 Weights;

uniform mat4 gBones[2];

void main(void) {
    int id0 = int(BoneIDs.x);
    int id1 = int(BoneIDs.y);
    int id2 = int(BoneIDs.z);
    int id3 = int(BoneIDs.w);

    mat4 BoneTransform =
          gBones[id0] * Weights.x +
          gBones[id1] * Weights.y +
          gBones[id2] * Weights.z +
          gBones[id3] * Weights.w;

    ex_Color = in_Color;

    gl_Position = MVP * BoneTransform * vec4(vVertex, 1.0);
}
</script>

In [ ]:
%%html
<script id="bones-fragment-shader" type="x-shader/x-fragment">

precision mediump float;
varying vec3 ex_Color;

void main(void) {
    
    gl_FragColor = vec4(ex_Color, 1.0);
    
}
</script>

### Objects

In [ ]:
%%html
<script>
class Background extends CGRAobject{
    constructor(glcontext){
        super(glcontext); // initialize the parent class
        
        this.numvertices = 6;

        // Two triangles forming a rectangle
        var vertices = [
            // First triangle
            -1.0, -1.0, 0.9,
             1.0, -1.0, 0.9,
            -1.0,  1.0, 0.9,

            // Second triangle
            -1.0,  1.0, 0.9,
             1.0, -1.0, 0.9,
             1.0,  1.0, 0.9
        ];

        var colors = [
            1.0, 0.0, 0.0,
            0.0, 1.0, 0.0,
            0.0, 0.0, 1.0,
            1.0, 0.0, 0.0,
            0.0, 1.0, 0.0,
            0.0, 0.0, 1.0];
        
        this.vertexbuffer=this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

        this.colorbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
    }
}

class BackgroundT extends Background{
    
    constructor(glcontext){
        super(glcontext);
            var texcoords = [
                0.0, 1.0,
                1.0, 1.0,
                0.0, 0.0,

                0.0, 0.0,
                1.0, 1.0,
                1.0, 0.0];
        
            this.texcoordbuffer = this.gl.createBuffer();
            this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.texcoordbuffer);    
            this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(texcoords), this.gl.STATIC_DRAW);
    }
        
    settexture(cgratex){
            this.textureid = cgratex.textureid;
    }
    
    drawit(viewMat, projectionMat){
        this.shaderprog.startUsing();
        this.texcoordsLocation = this.gl.getAttribLocation(this.shaderprog.shaderProgram,
                                                          "in_Tex_Coord");
       
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.texcoordbuffer);
        this.gl.vertexAttribPointer(this.texcoordsLocation, // Attribute location
                           2, // number of elements per attribute
                           this.gl.FLOAT,  // Type of elements
                           false,  // 
                           0, //2*Float32Array.BYTES_PER_ELEMENT, // size of a vertex in bytes 
                           0); // Offset from the begining of a single vertex to this attribute
        this.gl.enableVertexAttribArray(this.texcoordsLocation);
        this.gl.bindTexture(this.gl.TEXTURE_2D, this.textureid);
        this.texturelocation =  this.gl.getUniformLocation(this.shaderprog.shaderProgram, "uSampler");
        this.gl.uniform1i(this.textureLocation, 0);
        
        // the parent method does the rest
        super.drawit(viewMat,projectionMat);
    }    
}


</script>

In [ ]:
%%html

<script id="cubeT">
class cubeT extends CGRAobject{
    constructor(glcontext, color1=[1,0,0], color2=[0,0,1]) {
        super(glcontext);
        
        var vertices = [
          // Front face
            -0.5,-0.5, 0.5,
             0.5,-0.5, 0.5,
             0.5, 0.5, 0.5,
            -0.5, 0.5, 0.5,

            // Back face
             0.5,-0.5,-0.5,
            -0.5,-0.5,-0.5,
            -0.5, 0.5,-0.5,
             0.5, 0.5,-0.5,

            // Left face
            -0.5,-0.5,-0.5,
            -0.5,-0.5, 0.5,
            -0.5, 0.5, 0.5,
            -0.5, 0.5,-0.5,

            // Right face
             0.5,-0.5, 0.5,
             0.5,-0.5,-0.5,
             0.5, 0.5,-0.5,
             0.5, 0.5, 0.5,

            // Top face
            -0.5, 0.5, 0.5,
             0.5, 0.5, 0.5,
             0.5, 0.5,-0.5,
            -0.5, 0.5,-0.5,

            // Bottom face
            -0.5,-0.5,-0.5,
             0.5,-0.5,-0.5,
             0.5,-0.5, 0.5,
            -0.5,-0.5, 0.5
        ];

        var indices = [
          0,1,2, 0,2,3,      // front
          4,5,6, 4,6,7,      // back
          8,9,10, 8,10,11,   // left
          12,13,14, 12,14,15,// right
          16,17,18, 16,18,19,// top
          20,21,22, 20,22,23 // bottom
        ];

        var colors = [];
        for (let i = 0; i < vertices.length; i += 3) {//check x of every vertex
            
          //If x==-0.5 assign color1 otherwise its the other side and assign color2  
          const t = (vertices[i] + 0.5);
            
          colors.push(
            color1[0]*(1-t)+color2[0]*t,
            color1[1]*(1-t)+color2[1]*t,
            color1[2]*(1-t)+color2[2]*t
          );
        }
        
        var texcoords = [
            0,0, 1,0, 1,1, 0,1,
            0,0, 1,0, 1,1, 0,1,
            0,0, 1,0, 1,1, 0,1,
            0,0, 1,0, 1,1, 0,1,
            0,0, 1,0, 1,1, 0,1,
            0,0, 1,0, 1,1, 0,1
        ];
        
        var normals = [
            // Front face (0,0,1)
             0, 0, 1,
             0, 0, 1,
             0, 0, 1,
             0, 0, 1,

            // Back face (0,0,-1)
             0, 0,-1,
             0, 0,-1,
             0, 0,-1,
             0, 0,-1,

            // Left face (-1,0,0)
            -1, 0, 0,
            -1, 0, 0,
            -1, 0, 0,
            -1, 0, 0,

            // Right face (1,0,0)
             1, 0, 0,
             1, 0, 0,
             1, 0, 0,
             1, 0, 0,

            // Top face (0,1,0)
             0, 1, 0,
             0, 1, 0,
             0, 1, 0,
             0, 1, 0,

            // Bottom face (0,-1,0)
             0,-1, 0,
             0,-1, 0,
             0,-1, 0,
             0,-1, 0
        ];
        
        this.vertexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

        this.colorbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
        
        this.texcoordbuffer = this.gl.createBuffer();                           
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.texcoordbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(texcoords), this.gl.STATIC_DRAW);
        
        this.normalbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.normalbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(normals), this.gl.STATIC_DRAW);

        
        this.indexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ELEMENT_ARRAY_BUFFER, new Uint16Array(indices), this.gl.STATIC_DRAW);

        this.numIndices = indices.length;
    }
    
    setTexture(tex){
        this.texture = tex;      
    }
    
    draw(viewMat, projMat, shader, parentMat = glm.mat4(1.0)) {
        const gl = this.gl;
        shader.startUsing();

        const mvploc = gl.getUniformLocation(shader.shaderProgram, "MVP");
        const localT = parentMat['*'](this.modelMat);
        
        const normalloc = gl.getUniformLocation(shader.shaderProgram, "NormalMatrix");
        const modelloc = gl.getUniformLocation(shader.shaderProgram, "ModelMat");
        gl.uniformMatrix4fv(modelloc, false, localT.elements);
        
        // Inverse transpose
        let N = glm.transpose(glm.inverse(localT));

        // Upload to shader as float32array
        gl.uniformMatrix3fv(normalloc, false, new Float32Array([
              N.elements[0], N.elements[4], N.elements[8],
              N.elements[1], N.elements[5], N.elements[9],
              N.elements[2], N.elements[6], N.elements[10]
        ]));
        
        const MVP = projMat['*'](viewMat['*'](localT));
        gl.uniformMatrix4fv(mvploc, false, MVP.elements);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.vertexbuffer);
        const posLoc = gl.getAttribLocation(shader.shaderProgram, "in_Position");
        gl.vertexAttribPointer(posLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(posLoc);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.colorbuffer);
        const colLoc = gl.getAttribLocation(shader.shaderProgram, "in_Color");
        gl.vertexAttribPointer(colLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(colLoc);

        
        const texLoc = gl.getAttribLocation(shader.shaderProgram, "in_Tex_Coord");
        if (texLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.texcoordbuffer);
            gl.vertexAttribPointer(texLoc, 2, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(texLoc);
        }

        if (this.texture) {                                            
            gl.activeTexture(gl.TEXTURE0);
            gl.bindTexture(gl.TEXTURE_2D, this.texture.textureid);
            const samplerLoc = gl.getUniformLocation(shader.shaderProgram, "myTexture");
            if (samplerLoc !== -1) {
                gl.uniform1i(samplerLoc, 0); // usar TEXTURE0
            }
        }
        
        const normLoc = gl.getAttribLocation(shader.shaderProgram, "in_Normal");
        if (normLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.normalbuffer);
            gl.vertexAttribPointer(normLoc, 3, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(normLoc);
        }
        
        gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        gl.drawElements(gl.TRIANGLES, this.numIndices, gl.UNSIGNED_SHORT, 0);

        shader.stopUsing();
    }
}
</script>

In [ ]:
%%html

<script id="sphere">

class sphere extends CGRAobject {
    constructor(glcontext, color1=[1,0,0], color2=[0,0,1]) {
        super(glcontext);
        this.gl = glcontext;

        var nCircles = 12;  // number of circles
        var nVertices = 12;  // number of vertices per circle
        var radius = 0.5;

        var vertices = [];
        var indices = [];
        var colors = [];

        // Top vertice
        vertices.push(0.0, radius, 0.0);
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        const topIndex = 0;

        // Generate middle circles
        for (let i = 1; i < nCircles; i++) {
            
            var r = radius * Math.sin(i / nCircles * Math.PI);
            var y = radius * Math.cos(i / nCircles * Math.PI);
            
            for (let j = 0; j < nVertices; j++) {
                
                var x = r * Math.cos(j / nVertices * 2 * Math.PI);
                var z = r * Math.sin(j / nVertices * 2 * Math.PI);
                
                vertices.push(x, y, z);
                
                var t = j/nVertices; // based on x coordinate
                colors.push(
                color1[0] * (1 - t) + color2[0] * t,
                color1[1] * (1 - t) + color2[1] * t,
                color1[2] * (1 - t) + color2[2] * t
                );
            }
        }

        // Bottom vertice
        vertices.push(0.0, -radius, 0.0);
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        const bottomIndex = vertices.length / 3 - 1;

        // Indices
        // Top 
        for (let j = 1; j < nVertices; j++) {
            const next = (j % nVertices) +1;
            indices.push(topIndex, j, next);
        }

        // Middle stacks
        for (let i = 0; i < nCircles - 2; i++) {
            var start = 1 + i * nVertices;
            var next = start + nVertices;
            
            for (let j = 0; j < nVertices; j++) {
                const k = (j % nVertices) + 1;
            
                indices.push(start + j, next + j, next + k);
                indices.push(start + j, next + k, start + k);
            }
        }

        // Bottom 
        var lastRingStart = 1 + (nCircles - 2) * nVertices;
        for (let j = 0; j < nVertices; j++) {
            var next = (j % nVertices) + 1;
            indices.push(lastRingStart + j, bottomIndex, lastRingStart + next);
        }
        

        this.vertexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

        this.colorbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
        
        this.indexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ELEMENT_ARRAY_BUFFER, new Uint16Array(indices), this.gl.STATIC_DRAW);

        this.numIndices = indices.length;
    }

    draw(viewMat, projMat, shader, parentMat = glm.mat4(1.0)) {
        const gl = this.gl;
        shader.startUsing();

        const mvploc = gl.getUniformLocation(shader.shaderProgram, "MVP");
        const localT = parentMat['*'](this.modelMat);
        const MVP = projMat['*'](viewMat['*'](localT));
        gl.uniformMatrix4fv(mvploc, false, MVP.elements);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.vertexbuffer);
        const posLoc = gl.getAttribLocation(shader.shaderProgram, "in_Position");
        gl.vertexAttribPointer(posLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(posLoc);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.colorbuffer);
        const colLoc = gl.getAttribLocation(shader.shaderProgram, "in_Color");
        gl.vertexAttribPointer(colLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(colLoc);

        gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        gl.drawElements(gl.TRIANGLES, this.numIndices, gl.UNSIGNED_SHORT, 0);

        shader.stopUsing();
    }
}


</script>

In [ ]:
%%html

<script id="sphereT">

class sphereT extends CGRAobject {
    constructor(glcontext, color1=[1,0,0], color2=[0,0,1]) {
        super(glcontext);
        this.gl = glcontext;

        var nCircles = 12;  // number of circles
        var nVertices = 12;  // number of vertices per circle
        var radius = 0.5;

        var vertices = [];
        var indices = [];
        var colors = [];
        var texcoords = [];
        var normals = [];

        // Top vertice
        vertices.push(0.0, radius, 0.0);
        normals.push(0, 1, 0);
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        texcoords.push(0.5, 0.0);
        const topIndex = 0;

        // Generate middle circles
        for (let i = 1; i < nCircles; i++) {
            
            var r = radius * Math.sin(i / nCircles * Math.PI);
            var y = radius * Math.cos(i / nCircles * Math.PI);
            var v = i / nCircles;
            
            for (let j = 0; j < nVertices; j++) {
                
                var x = r * Math.cos(j / nVertices * 2 * Math.PI);
                var z = r * Math.sin(j / nVertices * 2 * Math.PI);
                
                vertices.push(x, y, z);
                let len = Math.sqrt(x*x + y*y + z*z);
                normals.push(x/len, y/len, z/len);
                
                var t = j/nVertices; // based on x coordinate
                colors.push(
                    color1[0] * (1 - t) + color2[0] * t,
                    color1[1] * (1 - t) + color2[1] * t,
                    color1[2] * (1 - t) + color2[2] * t
                );
                
                var u = j / nVertices;   
                texcoords.push(u, v);
            }
        }

        // Bottom vertice
        vertices.push(0.0, -radius, 0.0);
        normals.push(0, -1, 0);
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        texcoords.push(0.5, 1.0);
        const bottomIndex = vertices.length / 3 - 1;

        // Indices
        // Top 
        for (let j = 1; j < nVertices; j++) {
            const next = (j % nVertices) +1;
            indices.push(topIndex, j, next);
        }

        // Middle stacks
        for (let i = 0; i < nCircles - 2; i++) {
            var start = 1 + i * nVertices;
            var next = start + nVertices;
            
            for (let j = 0; j < nVertices; j++) {
                const k = (j % nVertices) + 1;
            
                indices.push(start + j, next + j, next + k);
                indices.push(start + j, next + k, start + k);
            }
        }

        // Bottom 
        var lastRingStart = 1 + (nCircles - 2) * nVertices;
        for (let j = 0; j < nVertices; j++) {
            var next = (j % nVertices) + 1;
            indices.push(lastRingStart + j, bottomIndex, lastRingStart + next);
        }
        

        this.vertexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

        this.colorbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
        
        // texcoords
        this.texcoordbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.texcoordbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(texcoords), this.gl.STATIC_DRAW);
        
        this.normalbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.normalbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(normals), this.gl.STATIC_DRAW);

        
        this.indexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ELEMENT_ARRAY_BUFFER, new Uint16Array(indices), this.gl.STATIC_DRAW);

        this.numIndices = indices.length;
    }
    
    setTexture(tex) {
        this.texture = tex;
    }

    draw(viewMat, projMat, shader, parentMat = glm.mat4(1.0)) {
        const gl = this.gl;
        shader.startUsing();

        const mvploc = gl.getUniformLocation(shader.shaderProgram, "MVP");
        const localT = parentMat['*'](this.modelMat);
        
        const normalloc = gl.getUniformLocation(shader.shaderProgram, "NormalMatrix");
        const modelloc = gl.getUniformLocation(shader.shaderProgram, "ModelMat");
        gl.uniformMatrix4fv(modelloc, false, localT.elements);
        
        // Inverse transpose
        let N = glm.transpose(glm.inverse(localT));

        // Upload to shader as float32array
        gl.uniformMatrix3fv(normalloc, false, new Float32Array([
              N.elements[0], N.elements[4], N.elements[8],
              N.elements[1], N.elements[5], N.elements[9],
              N.elements[2], N.elements[6], N.elements[10]
        ]));
        
        const MVP = projMat['*'](viewMat['*'](localT));
        gl.uniformMatrix4fv(mvploc, false, MVP.elements);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.vertexbuffer);
        const posLoc = gl.getAttribLocation(shader.shaderProgram, "in_Position");
        gl.vertexAttribPointer(posLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(posLoc);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.colorbuffer);
        const colLoc = gl.getAttribLocation(shader.shaderProgram, "in_Color");
        gl.vertexAttribPointer(colLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(colLoc);

        // texcoords
        const texLoc = gl.getAttribLocation(shader.shaderProgram, "in_Tex_Coord");
        if (texLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.texcoordbuffer);
            gl.vertexAttribPointer(texLoc, 2, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(texLoc);
        }

        // ligar textura ao sampler myTexture
        if (this.texture) {
            gl.activeTexture(gl.TEXTURE0);
            gl.bindTexture(gl.TEXTURE_2D, this.texture.textureid);
            const samplerLoc = gl.getUniformLocation(shader.shaderProgram, "myTexture");
            if (samplerLoc !== -1) {
                gl.uniform1i(samplerLoc, 0);
            }
        }
        
        const normLoc = gl.getAttribLocation(shader.shaderProgram, "in_Normal");
        if (normLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.normalbuffer);
            gl.vertexAttribPointer(normLoc, 3, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(normLoc);
        }
        
        gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        gl.drawElements(gl.TRIANGLES, this.numIndices, gl.UNSIGNED_SHORT, 0);

        shader.stopUsing();
    }
}


</script>

In [ ]:
%%html

<script id="cylinderT">
class cylinderT extends CGRAobject {
    constructor(glcontext, color1=[1,0,0], color2=[0,0,1]) {
        super(glcontext);

        const n = 12;
        const radius = 0.5;
        const height = 1.0;

        const vertices = [];
        const colors = [];
        const indices = [];
        const texcoords = [];
        const normals = [];
        
        //Top and base circle vertices

        for (let i = 0; i < n; i++) {

            var x = radius * Math.cos((i / n) * 2 * Math.PI);
            var z = radius * Math.sin((i / n) * 2 * Math.PI);
            vertices.push(x, -height / 2, z);

            var t = i/n; // Gradient along the vertices
            colors.push(
                color1[0] * (1 - t) + color2[0] * t,
                color1[1] * (1 - t) + color2[1] * t,
                color1[2] * (1 - t) + color2[2] * t
            );
            
            const u = i / n;
            const v = 0.0;
            texcoords.push(u, v);
            let len = Math.sqrt(x*x + z*z);
            normals.push(x/len, 0, z/len);
        }

        for (let i = 0; i < n; i++) {

            var x = radius * Math.cos((i / n) * 2 * Math.PI);
            var z = radius * Math.sin((i / n) * 2 * Math.PI);
            vertices.push(x, height / 2, z);

            var t = i/n; // Gradient along the vertices
            colors.push(
                color1[0] * (1 - t) + color2[0] * t,
                color1[1] * (1 - t) + color2[1] * t,
                color1[2] * (1 - t) + color2[2] * t
            );
            
            const u = i / n;
            const v = 1.0;
            texcoords.push(u, v);
            let len = Math.sqrt(x*x + z*z);
            normals.push(x/len, 0, z/len);
        }

        // Top and base center vertices
        var baseIndex = vertices.length / 3; //Divide by 3 because there are 3 vertices for each index
        vertices.push(0, -height / 2, 0);
        normals.push(0, -1, 0);
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        texcoords.push(0.5, 0.5);

        var topIndex = vertices.length / 3; //Divide by 3 because there are 3 vertices for each index
        vertices.push(0, height / 2, 0);
        normals.push(0, 1, 0);
         colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        texcoords.push(0.5, 0.5);


        //Side indices
        for (let i = 0; i < n; i++) {
            var next = (i + 1) % n; //The result is always the next number after i except in the last where we want 0 again
            var bottom1 = i;
            var bottom2 = next;
            var top1 = i + n;
            var top2 = next + n;

            // First triangle
            indices.push(bottom1, top1, bottom2);
            // Second triangle
            indices.push(bottom2, top1, top2);
        }

        // Top indices
        for (let i = 0; i < n; i++) {
            var next = (i + 1) % n; //The result is always the next number after i except in the last where we want 0 again
            indices.push(topIndex, i+n, next+n);
        }
        
        // Base indices
        for (let i = 0; i < n; i++) {
            var next = (i + 1) % n; //The result is always the next number after i except in the last where we want 0 again
            indices.push(baseIndex, next, i);
        }


        this.vertexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

        this.colorbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
        
        // texcoords
        this.texcoordbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.texcoordbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(texcoords), this.gl.STATIC_DRAW);
    
        this.normalbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.normalbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(normals), this.gl.STATIC_DRAW);

  
        this.indexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ELEMENT_ARRAY_BUFFER, new Uint16Array(indices), this.gl.STATIC_DRAW);

        this.numIndices = indices.length;
    }
    
    setTexture(tex) {
        this.texture = tex;
    }

    draw(viewMat, projMat, shader, parentMat = glm.mat4(1.0)) {
        const gl = this.gl;
        shader.startUsing();

        const mvploc = gl.getUniformLocation(shader.shaderProgram, "MVP");
        const localT = parentMat['*'](this.modelMat);
        
        const normalloc = gl.getUniformLocation(shader.shaderProgram, "NormalMatrix");
        const modelloc = gl.getUniformLocation(shader.shaderProgram, "ModelMat");
        gl.uniformMatrix4fv(modelloc, false, localT.elements);
        
        // Inverse transpose
        let N = glm.transpose(glm.inverse(localT));

        // Upload to shader as float32array
        gl.uniformMatrix3fv(normalloc, false, new Float32Array([
              N.elements[0], N.elements[4], N.elements[8],
              N.elements[1], N.elements[5], N.elements[9],
              N.elements[2], N.elements[6], N.elements[10]
        ]));
        
        const MVP = projMat['*'](viewMat['*'](localT));
        gl.uniformMatrix4fv(mvploc, false, MVP.elements);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.vertexbuffer);
        const posLoc = gl.getAttribLocation(shader.shaderProgram, "in_Position");
        gl.vertexAttribPointer(posLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(posLoc);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.colorbuffer);
        const colLoc = gl.getAttribLocation(shader.shaderProgram, "in_Color");
        gl.vertexAttribPointer(colLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(colLoc);

        // texcoords
        const texLoc = gl.getAttribLocation(shader.shaderProgram, "in_Tex_Coord");
        if (texLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.texcoordbuffer);
            gl.vertexAttribPointer(texLoc, 2, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(texLoc);
        }

        // texture
        if (this.texture) {
            gl.activeTexture(gl.TEXTURE0);
            gl.bindTexture(gl.TEXTURE_2D, this.texture.textureid);
            const samplerLoc = gl.getUniformLocation(shader.shaderProgram, "myTexture");
            if (samplerLoc !== -1) {
                gl.uniform1i(samplerLoc, 0);
            }
        }
        
        const normLoc = gl.getAttribLocation(shader.shaderProgram, "in_Normal");
        if (normLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.normalbuffer);
            gl.vertexAttribPointer(normLoc, 3, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(normLoc);
        }
        
        gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        gl.drawElements(gl.TRIANGLES, this.numIndices, gl.UNSIGNED_SHORT, 0);

        shader.stopUsing();
    }
}
</script>


In [ ]:
%%html

<script id="cone">

class cone extends CGRAobject {
    constructor(glcontext, color1=[1,0,0], color2=[0,0,1]) {
        super(glcontext);
        this.gl = glcontext;
        
        var n = 12;
        var radius = 0.5;
        var height = 1.0;
        var vertices = [];
        var indices = [];
        var colors = [];
        
        // top center vertice
        vertices.push(0.0, height/2.0, 0.0);
        
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        
        // base vertices
        for (let i = 0; i < n; i++) {
            var x = radius * Math.cos((i / n) * 2 * Math.PI);
            var y = -height/2;
            var z = radius * Math.sin((i / n) * 2 * Math.PI);
            vertices.push(x, y, z);
            
          var t = i/n; //Gradient along the vertices
            colors.push(
                color1[0] * (1 - t) + color2[0] * t,
                color1[1] * (1 - t) + color2[1] * t,
                color1[2] * (1 - t) + color2[2] * t
            );
        }
        
        // base center vertice
        vertices.push(0.0, -height/2.0, 0.0);
        
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        
        var topIndex = 0;
        var baseIndex = n + 1;
        
        // Indices
        for (let i = 1; i <= n; i++) {
            var next = (i % n) + 1; //The result is always the next number after i except in the last where we want 0 again
            indices.push(topIndex, i, next);
        }
        
        for (let i = 1; i <= n; i++) {
            var next = (i % n) + 1; //The result is always the next number after i except in the last where we want 0 again
            indices.push(baseIndex, next, i);
        }

        
        this.vertexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

        this.colorbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
        
        this.indexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ELEMENT_ARRAY_BUFFER, new Uint16Array(indices), this.gl.STATIC_DRAW);

        this.numIndices = indices.length;
    }
    
    draw(viewMat, projMat, shader, parentMat = glm.mat4(1.0)) {
        const gl = this.gl;
        shader.startUsing();

        const mvploc = gl.getUniformLocation(shader.shaderProgram, "MVP");
        const localT = parentMat['*'](this.modelMat);
        const MVP = projMat['*'](viewMat['*'](localT));
        gl.uniformMatrix4fv(mvploc, false, MVP.elements);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.vertexbuffer);
        const posLoc = gl.getAttribLocation(shader.shaderProgram, "in_Position");
        gl.vertexAttribPointer(posLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(posLoc);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.colorbuffer);
        const colLoc = gl.getAttribLocation(shader.shaderProgram, "in_Color");
        gl.vertexAttribPointer(colLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(colLoc);

        gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        gl.drawElements(gl.TRIANGLES, this.numIndices, gl.UNSIGNED_SHORT, 0);

        shader.stopUsing();
    }

}

</script>

In [ ]:
%%html

<script id="coneT">

class coneT extends CGRAobject {
    constructor(glcontext, color1=[1,0,0], color2=[0,0,1]) {
        super(glcontext);
        this.gl = glcontext;
        
        var n = 12;
        var radius = 0.5;
        var height = 1.0;
        var vertices = [];
        var indices = [];
        var colors = [];
        var texcoords  = [];
        var normals = [];
        var slope = radius / height;
        
        // top center vertice
        vertices.push(0.0, height/2.0, 0.0);
        normals.push(0, 1, 0);
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        texcoords.push(0.5, 1.0);
        
        // base vertices
        for (let i = 0; i < n; i++) {
            var x = radius * Math.cos((i / n) * 2 * Math.PI);
            var y = -height/2;
            var z = radius * Math.sin((i / n) * 2 * Math.PI);
            vertices.push(x, y, z);
            
            var t = i/n; //Gradient along the vertices
            colors.push(
                color1[0] * (1 - t) + color2[0] * t,
                color1[1] * (1 - t) + color2[1] * t,
                color1[2] * (1 - t) + color2[2] * t
            );

            // mapeamento de textura para a lateral
            var u = i / n;
            var v = 0.0;
            texcoords.push(u, v);
            
            let nx = x;
            let ny = slope * radius;  // upward tilt
            let nz = z;

            let len = Math.sqrt(nx*nx + ny*ny + nz*nz);
            normals.push(nx/len, ny/len, nz/len);
        }
        
        // base center vertice
        vertices.push(0.0, -height/2.0, 0.0);
        normals.push(0, -1, 0);
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        texcoords.push(0.5, 0.5);
        
        var topIndex = 0;
        var baseIndex = n + 1;
        
        // Indices
        for (let i = 1; i <= n; i++) {
            var next = (i % n) + 1; //The result is always the next number after i except in the last where we want 0 again
            indices.push(topIndex, i, next);
        }
        
        for (let i = 1; i <= n; i++) {
            var next = (i % n) + 1; //The result is always the next number after i except in the last where we want 0 again
            indices.push(baseIndex, next, i);
        }

        
        this.vertexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

        this.colorbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
        
        // texcoords
        this.texcoordbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.texcoordbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(texcoords), this.gl.STATIC_DRAW);
  
        this.normalbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.normalbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(normals), this.gl.STATIC_DRAW);

  
        this.indexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ELEMENT_ARRAY_BUFFER, new Uint16Array(indices), this.gl.STATIC_DRAW);

        this.numIndices = indices.length;
    }
    
    setTexture(tex) {
        this.texture = tex;
    }
    
    draw(viewMat, projMat, shader, parentMat = glm.mat4(1.0)) {
        const gl = this.gl;
        shader.startUsing();

        const mvploc = gl.getUniformLocation(shader.shaderProgram, "MVP");
        const localT = parentMat['*'](this.modelMat);
        
        const normalloc = gl.getUniformLocation(shader.shaderProgram, "NormalMatrix");
        const modelloc = gl.getUniformLocation(shader.shaderProgram, "ModelMat");
        gl.uniformMatrix4fv(modelloc, false, localT.elements);
        
        // Inverse transpose
        let N = glm.transpose(glm.inverse(localT));

        // Upload to shader as float32array
        gl.uniformMatrix3fv(normalloc, false, new Float32Array([
              N.elements[0], N.elements[4], N.elements[8],
              N.elements[1], N.elements[5], N.elements[9],
              N.elements[2], N.elements[6], N.elements[10]
        ]));
        
        const MVP = projMat['*'](viewMat['*'](localT));
        gl.uniformMatrix4fv(mvploc, false, MVP.elements);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.vertexbuffer);
        const posLoc = gl.getAttribLocation(shader.shaderProgram, "in_Position");
        gl.vertexAttribPointer(posLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(posLoc);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.colorbuffer);
        const colLoc = gl.getAttribLocation(shader.shaderProgram, "in_Color");
        gl.vertexAttribPointer(colLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(colLoc);

        // texcoords
        const texLoc = gl.getAttribLocation(shader.shaderProgram, "in_Tex_Coord");
        if (texLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.texcoordbuffer);
            gl.vertexAttribPointer(texLoc, 2, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(texLoc);
        }

        // texture
        if (this.texture) {
            gl.activeTexture(gl.TEXTURE0);
            gl.bindTexture(gl.TEXTURE_2D, this.texture.textureid);
            const samplerLoc = gl.getUniformLocation(shader.shaderProgram, "myTexture");
            if (samplerLoc !== -1) {
                gl.uniform1i(samplerLoc, 0);
            }
        }
        
        const normLoc = gl.getAttribLocation(shader.shaderProgram, "in_Normal");
        if (normLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.normalbuffer);
            gl.vertexAttribPointer(normLoc, 3, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(normLoc);
        }
        
        gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        gl.drawElements(gl.TRIANGLES, this.numIndices, gl.UNSIGNED_SHORT, 0);

        shader.stopUsing();
    }

}

</script>

In [ ]:
%%html

<script id="diskT">
class diskT extends CGRAobject {
    constructor(glcontext, color1=[1,0,0], color2=[0,0,1], n = 12) {
        super(glcontext);

        var radius = 0.5;

        var vertices = [0.0, 0.0, 0.0]; //Center vertex
        var texcoords = [];
        var normals = [];
        normals.push(0, 1, 0); 
        
        var colors = [];
        // Center color 
        colors.push(
            (color1[0] + color2[0]) / 2,
            (color1[1] + color2[1]) / 2,
            (color1[2] + color2[2]) / 2
        );
        texcoords.push(0.5, 0.5);

        for (var i = 0; i < n; i++) {
            
            var x = radius * Math.cos((i / n) * 2 * Math.PI);
            var y = 0.0;
            var z = radius * Math.sin((i / n) * 2 * Math.PI);
            vertices.push(x, y, z);

            var t = i/n; // Gradient along the vertices
            colors.push(
                color1[0] * (1 - t) + color2[0] * t,
                color1[1] * (1 - t) + color2[1] * t,
                color1[2] * (1 - t) + color2[2] * t
            );
            
            var u = x + 0.5;
            var v = z + 0.5;
            texcoords.push(u, v);
            
            normals.push(0, 1, 0); 
        }

        var indices = [];
        for (let i = 1; i <= n; i++) {
            var next = (i % n) + 1; //The result is always the next number after i except in the last where we want 1 again
            indices.push(0, i, next);
        }

        this.vertexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

        this.colorbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
        
        // texcoords
        this.texcoordbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.texcoordbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(texcoords), this.gl.STATIC_DRAW);
      
        this.normalbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.normalbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(normals), this.gl.STATIC_DRAW);

  
        this.indexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ELEMENT_ARRAY_BUFFER, new Uint16Array(indices), this.gl.STATIC_DRAW);

        this.numIndices = indices.length;
    }

    setTexture(tex) {
        this.texture = tex;
    }
    
    draw(viewMat, projMat, shader, parentMat = glm.mat4(1.0)) {
        const gl = this.gl;
        shader.startUsing();

        const mvploc = gl.getUniformLocation(shader.shaderProgram, "MVP");
        const localT = parentMat['*'](this.modelMat);
        
        const normalloc = gl.getUniformLocation(shader.shaderProgram, "NormalMatrix");
        const modelloc = gl.getUniformLocation(shader.shaderProgram, "ModelMat");
        gl.uniformMatrix4fv(modelloc, false, localT.elements);
        
        // Inverse transpose
        let N = glm.transpose(glm.inverse(localT));

        // Upload to shader as float32array
        gl.uniformMatrix3fv(normalloc, false, new Float32Array([
              N.elements[0], N.elements[4], N.elements[8],
              N.elements[1], N.elements[5], N.elements[9],
              N.elements[2], N.elements[6], N.elements[10]
        ]));
        
        const MVP = projMat['*'](viewMat['*'](localT));
        gl.uniformMatrix4fv(mvploc, false, MVP.elements);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.vertexbuffer);
        const posLoc = gl.getAttribLocation(shader.shaderProgram, "in_Position");
        gl.vertexAttribPointer(posLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(posLoc);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.colorbuffer);
        const colLoc = gl.getAttribLocation(shader.shaderProgram, "in_Color");
        gl.vertexAttribPointer(colLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(colLoc);

        // texcoords
        const texLoc = gl.getAttribLocation(shader.shaderProgram, "in_Tex_Coord");
        if (texLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.texcoordbuffer);
            gl.vertexAttribPointer(texLoc, 2, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(texLoc);
        }

        // texture
        if (this.texture) {
            gl.activeTexture(gl.TEXTURE0);
            gl.bindTexture(gl.TEXTURE_2D, this.texture.textureid);
            const samplerLoc = gl.getUniformLocation(shader.shaderProgram, "myTexture");
            if (samplerLoc !== -1) {
                gl.uniform1i(samplerLoc, 0);
            }
        }
        
        const normLoc = gl.getAttribLocation(shader.shaderProgram, "in_Normal");
        if (normLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.normalbuffer);
            gl.vertexAttribPointer(normLoc, 3, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(normLoc);
        }
        
        gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        gl.drawElements(gl.TRIANGLES, this.numIndices, gl.UNSIGNED_SHORT, 0);

        shader.stopUsing();
    }
}
</script>


In [ ]:
%%html

<script id="pyramidT">
class pyramidT extends CGRAobject {
      constructor(glcontext, color1=[1,0,0], color2=[0,0,1]) {
        super(glcontext);


        var vertices = [

          -0.5, -0.5, -0.5, // 0
           0.5, -0.5, -0.5, // 1
           0.5, -0.5,  0.5, // 2
          -0.5, -0.5,  0.5, // 3

           0.0,  0.5,  0.0  // 4
        ];

        var indices = [
          // sides
          0, 1, 4, //Left side
          1, 2, 4,
          2, 3, 4,
          3, 0, 4,

          // base
          0, 1, 2,
          0, 2, 3
        ];

        var normals = [];
        var texcoords = [];
        var colors = [];
        for (let i = 0; i < vertices.length; i += 3) {//check x of every vertex
          const x = vertices[i];
          const y = vertices[i+1];
          const z = vertices[i+2];
                      
            
          if (y === 0.5) {
              normals.push(0, 1, 0);
          }                                            
                                                      
          //If x==-0.5 assign color1 otherwise assign color2  
          const t = (vertices[i] + 0.5);

          colors.push(
            color1[0]*(1-t)+color2[0]*t,
            color1[1]*(1-t)+color2[1]*t,
            color1[2]*(1-t)+color2[2]*t
          );
                                                      
          const u = x + 0.5;
          const v = z + 0.5;
          texcoords.push(u, v);
                                                      
            let nx = x;
            let ny = 0.25;
            let nz = z;

            let len = Math.hypot(nx, ny, nz);
            nx /= len;
            ny /= len;
            nz /= len;

            normals.push(nx, ny, nz);
        }

        this.vertexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

        this.colorbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);    
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
        
        // texcoords
        this.texcoordbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.texcoordbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(texcoords), this.gl.STATIC_DRAW);  
          
        this.normalbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.normalbuffer);
        this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(normals), this.gl.STATIC_DRAW);

  
        this.indexbuffer = this.gl.createBuffer();
        this.gl.bindBuffer(this.gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        // as JS stores everything in 64 bit format and GL expects 32bits...
        this.gl.bufferData(this.gl.ELEMENT_ARRAY_BUFFER, new Uint16Array(indices), this.gl.STATIC_DRAW);

        this.numIndices = indices.length;
  }
    
    setTexture(tex){
        this.texture = tex;
    }
    
  draw(viewMat, projMat, shader, parentMat = glm.mat4(1.0)) {
        const gl = this.gl;
        shader.startUsing();

        const mvploc = gl.getUniformLocation(shader.shaderProgram, "MVP");
        const localT = parentMat['*'](this.modelMat);
      
        const normalloc = gl.getUniformLocation(shader.shaderProgram, "NormalMatrix");
        const modelloc = gl.getUniformLocation(shader.shaderProgram, "ModelMat");
        gl.uniformMatrix4fv(modelloc, false, localT.elements);
        
        // Inverse transpose
        let N = glm.transpose(glm.inverse(localT));

        // Upload to shader as float32array
        gl.uniformMatrix3fv(normalloc, false, new Float32Array([
              N.elements[0], N.elements[4], N.elements[8],
              N.elements[1], N.elements[5], N.elements[9],
              N.elements[2], N.elements[6], N.elements[10]
        ]));
      
        const MVP = projMat['*'](viewMat['*'](localT));
        gl.uniformMatrix4fv(mvploc, false, MVP.elements);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.vertexbuffer);
        const posLoc = gl.getAttribLocation(shader.shaderProgram, "in_Position");
        gl.vertexAttribPointer(posLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(posLoc);

        gl.bindBuffer(gl.ARRAY_BUFFER, this.colorbuffer);
        const colLoc = gl.getAttribLocation(shader.shaderProgram, "in_Color");
        gl.vertexAttribPointer(colLoc, 3, gl.FLOAT, false, 0, 0);
        gl.enableVertexAttribArray(colLoc);

        // texcoords
        const texLoc = gl.getAttribLocation(shader.shaderProgram, "in_Tex_Coord");
        if (texLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.texcoordbuffer);
            gl.vertexAttribPointer(texLoc, 2, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(texLoc);
        }

        // texture
        if (this.texture) {
            gl.activeTexture(gl.TEXTURE0);
            gl.bindTexture(gl.TEXTURE_2D, this.texture.textureid);
            const samplerLoc = gl.getUniformLocation(shader.shaderProgram, "myTexture");
            if (samplerLoc !== -1) {
                gl.uniform1i(samplerLoc, 0);
            }
        }
      
      
        const normLoc = gl.getAttribLocation(shader.shaderProgram, "in_Normal");
        if (normLoc !== -1) {
            gl.bindBuffer(gl.ARRAY_BUFFER, this.normalbuffer);
            gl.vertexAttribPointer(normLoc, 3, gl.FLOAT, false, 0, 0);
            gl.enableVertexAttribArray(normLoc);
        }

      
        gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
        gl.drawElements(gl.TRIANGLES, this.numIndices, gl.UNSIGNED_SHORT, 0);

        shader.stopUsing();
    }
}
</script>


In [ ]:
%%html
<script id="cylinderMesh">
class cylinderMesh extends CGRAobject {
  constructor(glcontext, radialSeg=12, stacks=10, radius=0.5, height=1.0, color1=[1,0,0], color2=[0,0,1]) {
    super(glcontext);

    const vertices = [];
    const normals = [];
    const texcoords = [];
    const colors = [];
    const boneIDs = [];
    const weights = [];
    const indices = [];

    for (let j = 0; j <= stacks; j++) {
      const v = j / stacks;              // 0..1
      const y = -height/2 + v*height;

      // cor desta stack (gradiente vertical)
      const r = color1[0]*(1-v) + color2[0]*v;
      const g = color1[1]*(1-v) + color2[1]*v;
      const b = color1[2]*(1-v) + color2[2]*v;

      for (let i = 0; i <= radialSeg; i++) {
        const u = i / radialSeg;
        const ang = u * 2.0 * Math.PI;

        const x = radius * Math.cos(ang);
        const z = radius * Math.sin(ang);

        vertices.push(x, y, z);
        normals.push(Math.cos(ang), 0.0, Math.sin(ang));
        texcoords.push(u, v);

        colors.push(r, g, b);

        // 2 bones
        let w1 = 0.0;
        if (v <= 0.4) w1 = 0.0;
        else if (v >= 0.6) w1 = 1.0;
        else w1 = (v - 0.4) / (0.6 - 0.4);
        const w0 = 1.0 - w1;

        boneIDs.push(0.0, 1.0, 0.0, 0.0);
        weights.push(w0, w1,  0.0, 0.0);
      }
    }

    const row = radialSeg + 1;
    for (let j = 0; j < stacks; j++) {
      for (let i = 0; i < radialSeg; i++) {
        const a = j*row + i;
        const b = a + row;
        indices.push(a, b, a+1);
        indices.push(a+1, b, b+1);
      }
    }

    const gl = this.gl;

    this.vertexbuffer = gl.createBuffer();
    gl.bindBuffer(gl.ARRAY_BUFFER, this.vertexbuffer);
    gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(vertices), gl.STATIC_DRAW);

    this.normalbuffer = gl.createBuffer();
    gl.bindBuffer(gl.ARRAY_BUFFER, this.normalbuffer);
    gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(normals), gl.STATIC_DRAW);

    this.texcoordbuffer = gl.createBuffer();
    gl.bindBuffer(gl.ARRAY_BUFFER, this.texcoordbuffer);
    gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(texcoords), gl.STATIC_DRAW);

    this.colorbuffer = gl.createBuffer();
    gl.bindBuffer(gl.ARRAY_BUFFER, this.colorbuffer);
    gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(colors), gl.STATIC_DRAW);

    this.boneIDbuffer = gl.createBuffer();
    gl.bindBuffer(gl.ARRAY_BUFFER, this.boneIDbuffer);
    gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(boneIDs), gl.STATIC_DRAW);

    this.weightbuffer = gl.createBuffer();
    gl.bindBuffer(gl.ARRAY_BUFFER, this.weightbuffer);
    gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(weights), gl.STATIC_DRAW);

    this.indexbuffer = gl.createBuffer();
    gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
    gl.bufferData(gl.ELEMENT_ARRAY_BUFFER, new Uint16Array(indices), gl.STATIC_DRAW);

    this.numIndices = indices.length;

    this.bones = [ glm.mat4(1.0), glm.mat4(1.0) ];
  }

  setBoneMatrix(i, M) { this.bones[i] = M; }

  setBoneRotation(i, angleRad, axis, pivot=glm.vec3(0,0,0)) {
    const T1 = glm.translate(glm.mat4(1.0), pivot);
    const R  = glm.toMat4(glm.angleAxis(angleRad, axis));
    const T2 = glm.translate(glm.mat4(1.0), glm.vec3(-pivot.x, -pivot.y, -pivot.z));
    this.bones[i] = T1['*'](R)['*'](T2);
  }

  draw(viewMat, projMat, shader, parentMat = glm.mat4(1.0)) {
    const gl = this.gl;
    shader.startUsing();

    const mvploc = gl.getUniformLocation(shader.shaderProgram, "MVP");
    const localT = parentMat['*'](this.modelMat);
    const MVP = projMat['*'](viewMat['*'](localT));
    gl.uniformMatrix4fv(mvploc, false, MVP.elements);
      
    const bonesLoc = gl.getUniformLocation(shader.shaderProgram, "gBones[0]");
    gl.uniformMatrix4fv(bonesLoc, false, new Float32Array([
      ...this.bones[0].elements,
      ...this.bones[1].elements
    ]));

    const bindAttr = (buf, name, size) => {
      const loc = gl.getAttribLocation(shader.shaderProgram, name);
      if (loc === -1) return;
      gl.bindBuffer(gl.ARRAY_BUFFER, buf);
      gl.vertexAttribPointer(loc, size, gl.FLOAT, false, 0, 0);
      gl.enableVertexAttribArray(loc);
    };

    bindAttr(this.vertexbuffer,   "vVertex",   3);
    bindAttr(this.normalbuffer,   "vNormal",   3);
    bindAttr(this.texcoordbuffer, "vTexCoords",2);
    bindAttr(this.colorbuffer,    "in_Color",  3);
    bindAttr(this.boneIDbuffer,   "BoneIDs",   4);
    bindAttr(this.weightbuffer,   "Weights",   4);

    gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, this.indexbuffer);
    gl.drawElements(gl.TRIANGLES, this.numIndices, gl.UNSIGNED_SHORT, 0);

    shader.stopUsing();
  }
}
</script>


In [ ]:
%%html
<script id="human">

class human extends CGRAobject {
    constructor(glcontext, color1 = [1,0,0], color2 = [0, 0, 1]){
        super(glcontext);
        
        this.torso = new cone(this.gl, color1, color2);     
        this.head = new sphere(this.gl, color1, color2);
        
        this.armL = new cylinderMesh(this.gl, 12, 10, 0.3, 2.0, color1, color2);
        this.armR = new cylinderMesh(this.gl, 12, 10, 0.3, 2.0, color1, color2);
        
        this.create_obj();
    }
    
    setShaders(shaderprog, shaderBones){       
        this.shader = shaderprog;
        this.shaderBones = shaderBones;
        
        this.torso.setShader(shaderprog);
        this.head.setShader(shaderprog);
        
        this.armL.setShader(shaderBones);
        this.armR.setShader(shaderBones);
    }
    
    create_obj(){
        
        // torso
        let rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,0.0,0.0)));
        let scale = glm.scale(glm.mat4(1.0), glm.vec3(2.0, 2.5, 1.5));
        let pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.25, 0.0));
        let transform = pos['*'](rot)['*'](scale);
        this.torso.setModelTransformation(transform);
        
        // head
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,0.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(1.5, 1.5, 1.5));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 2.75, 0.0));
        transform = pos['*'](rot)['*'](scale);
        this.head.setModelTransformation(transform);
        
        // armL
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,0.0,1.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(0.75, 1.0, 0.75));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(-1.0, 1.5, 0.0));
        transform = pos['*'](rot)['*'](scale);
        this.armL.setModelTransformation(transform);
        
        // armR
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,0.0,-1.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(0.75, 1.0, 0.75));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(1.0, 1.5, 0.0));
        transform = pos['*'](rot)['*'](scale);
        this.armR.setModelTransformation(transform);
        
    }
    
    draw(viewM = glm.mat4(1.0), projectionM = glm.mat4(1.0), parentM = glm.mat4(1.0), time = 0){
        const base = parentM['*'](this.modelMat);

        this.torso.draw(viewM, projectionM, this.shader, base);
        this.head.draw(viewM, projectionM, this.shader, base);

        let angle = 0.7 * Math.sin(time * 1.5);
        
        [this.armL, this.armR].forEach((arm, index) => {
            arm.setBoneMatrix(0, glm.mat4(1.0));
            arm.setBoneRotation(1, angle, glm.vec3(0, 0, 1), glm.vec3(0, 0, 0));
            
            arm.draw(viewM, projectionM, this.shaderBones, base);
        });
     }

}


</script>

In [ ]:
%%html
<script id="locomotiveEngine">

class locomotiveEngine extends CGRAobject {
    constructor(glcontext, color1 = [1,0,0], color2 = [0, 0, 1], mode=0){
        super(glcontext);
        
        this.color1 = color1;
        this.color2 = color2;
        
        this.cabine = new cubeT(this.gl, color1, color2);
        this.lower_body = new cubeT(this.gl, color1, color2);

        this.engine = new cylinderT(this.gl, color1, color2);
        this.chimney = new cylinderT(this.gl, color1, color2);
        this.nose = new cylinderT(this.gl, color1, color2);
        this.cabine_roof = new cylinderT(this.gl, color1, color2);

        this.chimney_top = new coneT(this.gl, color1, color2);
        
        this.coolers = [];
        this.swheels = [];
        this.bwheels = [];
        
        this.mode = mode;
        
        this.create_obj();
    }
    
    setShader(shaderprog){
        this.shader = shaderprog;
        
        this.cabine.setShader(shaderprog);
        this.lower_body.setShader(shaderprog);
        this.engine.setShader(shaderprog);
        this.chimney.setShader(shaderprog);
        this.nose.setShader(shaderprog);
        this.cabine_roof.setShader(shaderprog);
        this.chimney_top.setShader(shaderprog);
        
        for (let c of this.coolers) c.setShader(shaderprog);
        for (let w of this.swheels) w.setShader(shaderprog);
        for (let b of this.bwheels) b.setShader(shaderprog);
    }
    
    setTexture(bodyTex, wheelTex) {
        this.bodyTex  = bodyTex;
        this.wheelTex = wheelTex;

        this.cabine.setTexture(bodyTex);
        this.lower_body.setTexture(bodyTex);
        this.engine.setTexture(bodyTex);
        this.chimney.setTexture(bodyTex);
        this.nose.setTexture(bodyTex);
        this.cabine_roof.setTexture(bodyTex);
        this.chimney_top.setTexture(bodyTex);

        for (let c of this.coolers) {
            c.setTexture(bodyTex);
        }

        for (let w of this.swheels) {
            w.setTexture(wheelTex);
        }
        
        for (let b of this.bwheels) {
            b.setTexture(wheelTex);
        }
    }
    
    create_obj(){
        const c1 = this.color1;
        const c2 = this.color2;
        
        // lower_body
        let rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,1.0,0.0)));
        let scale = glm.scale(glm.mat4(1.0), glm.vec3(6.0, 2.0, 1.0));
        let pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 0.5, 0.0));
        let transform = pos['*'](rot)['*'](scale);
        this.lower_body.setModelTransformation(transform);

        // cabine
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,1.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(2.0, 1.5, 2.0));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.75, -2.0));
        transform = pos['*'](rot)['*'](scale);
        this.cabine.setModelTransformation(transform);

        // engine
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(1.0,0.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(2.25, 4.0, 2.25));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.5, 1.0));
        transform = pos['*'](rot)['*'](scale);
        this.engine.setModelTransformation(transform);

        // chimney
        rot = glm.toMat4(glm.angleAxis(glm.radians(180.0),glm.vec3(1.0,0.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(0.75, 1.0, 0.75));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 2.9, 2.5));
        transform = pos['*'](rot)['*'](scale);
        this.chimney.setModelTransformation(transform);

        // chimney top
        rot = glm.toMat4(glm.angleAxis(glm.radians(180.0),glm.vec3(1.0,0.0,0.0)));
        scale = glm.mat4(glm.mat3(0.5));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 3.2, 2.5));
        transform = pos['*'](rot)['*'](scale);
        this.chimney_top.setModelTransformation(transform);

        // small wheels
        const smallWheelPositions = [
            glm.vec3(0.7, -0.25, 2.5),   // right front
            glm.vec3(0.7, -0.25, 1.25),   // right back
            glm.vec3(-0.7, -0.25, 2.5),  // left front
            glm.vec3(-0.7, -0.25, 1.25)   // left back
        ];

        this.swheels = [];

        for (let i = 0; i < smallWheelPositions.length; i++) {
            const isRightSide = smallWheelPositions[i].x > 0;
        
            const color1 = isRightSide ? c1 : c2;
            const color2 = isRightSide ? c2 : c1;
            
            var wheel = new cylinderT(this.gl, color1, color2);
            
            if (this.mode==0){
                wheel = new cylinderT(this.gl, color1, color2);
            }else if (this.mode==1){
                wheel = new sphereT(this.gl, color1, color2);
            }else if (this.mode==2){
                wheel = new cubeT(this.gl, color1, color2);
            }
            
            const rot = glm.toMat4(glm.angleAxis(glm.radians(90.0), glm.vec3(0.0, 0.0, 1.0)));
            const scale = glm.scale(glm.mat4(1.0), glm.vec3(1.0, 0.5, 1.0));
            const pos = glm.translate(glm.mat4(1.0), smallWheelPositions[i]);
            const baseTransform = pos['*'](rot)['*'](scale);
            
            wheel.baseTransform = baseTransform;
            
            this.swheels.push(wheel);
        }

        // big wheel        
        const bigWheelPositions = [
            glm.vec3(-0.75, 0.15, -0.5),
            glm.vec3(-0.75, 0.15, -2.25),
            glm.vec3(0.75, 0.15, -0.5),
            glm.vec3(0.75, 0.15, -2.25)
        ];

        this.bwheels = [];

        for (let i = 0; i < bigWheelPositions.length; i++) {
            const isRightSide = bigWheelPositions[i].x > 0;
            
            const color1 = isRightSide ? c1 : c2;
            const color2 = isRightSide ? c2 : c1;
            
            var wheel = new cylinderT(this.gl, color1, color2);
            
            if (this.mode==0){
                wheel = new cylinderT(this.gl, color1, color2);
            }else if (this.mode==1){
                wheel = new sphereT(this.gl, color1, color2);
            }else if (this.mode==2){
                wheel = new cubeT(this.gl, color1, color2);
            }
            
            rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,0.0,1.0)));
            scale = glm.scale(glm.mat4(1.0), glm.vec3(1.75, 0.5, 1.75));
            const pos = glm.translate(glm.mat4(1.0), bigWheelPositions[i]);
            const baseTransform = pos['*'](rot)['*'](scale);
            
            wheel.baseTransform = baseTransform;
            
            this.bwheels.push(wheel);
        }

        // cabine roof
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(1.0,0.0,0.0)));
        scale = glm.mat4(glm.mat3(2.0));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 2.3, -2.0));
        transform = pos['*'](rot)['*'](scale);
        this.cabine_roof.setModelTransformation(transform);

        // coolers
        const coolerPositions = [
            glm.vec3(0.6, 0.75, 1.75),
            glm.vec3(-0.6, 0.75, 1.75),
        ];

        this.coolers = [];

        for (let i = 0; i < coolerPositions.length; i++) {
            const cooler = new cylinderT(this.gl, c1, c2);
            rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(1.0,0.0,0.0)));
            scale = glm.scale(glm.mat4(1.0), glm.vec3(0.75, 2.5, 0.75));
            const pos = glm.translate(glm.mat4(1.0), coolerPositions[i]);
            const transform = pos['*'](rot)['*'](scale);
            cooler.setModelTransformation(transform);
            this.coolers.push(cooler);
        }

        // nose
        rot = glm.toMat4(glm.angleAxis(glm.radians(270.0),glm.vec3(1.0,0.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(0.75, 0.25, 0.75));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.5, 3.0));
        transform = pos['*'](rot)['*'](scale);
        this.nose.setModelTransformation(transform);
    }

    
    draw(viewM = glm.mat4(1.0), projectionM = glm.mat4(1.0), parentM = glm.mat4(1.0), counter = 0){
        
        this.lower_body.draw(viewM, projectionM, this.shader, parentM);
        this.cabine.draw(viewM, projectionM, this.shader, parentM);
        this.engine.draw(viewM, projectionM, this.shader, parentM);
        this.chimney.draw(viewM, projectionM, this.shader, parentM);
        this.chimney_top.draw(viewM, projectionM, this.shader, parentM);
        this.nose.draw(viewM, projectionM, this.shader, parentM);
        this.cabine_roof.draw(viewM, projectionM, this.shader, parentM);
        
        for (let c of this.coolers) {
            c.draw(viewM, projectionM, this.shader, parentM);
        }
        
        for (let w of this.swheels) {
            const angle = (counter % (2 * Math.PI)) * 2;
            const spin = glm.toMat4(glm.angleAxis(angle, glm.vec3(0.0,1.0,0.0)));
            w.setModelTransformation(w.baseTransform['*'](spin));
            w.draw(viewM, projectionM, this.shader, parentM);
        }
        
        for (let b of this.bwheels) {
            const angle = (counter % (2 * Math.PI)) * 2;
            const spin = glm.toMat4(glm.angleAxis(angle, glm.vec3(0.0,1.0,0.0)));
            b.setModelTransformation(b.baseTransform['*'](spin));
            b.draw(viewM, projectionM, this.shader, parentM);
        }
    }
}


</script>


In [ ]:
%%html
<script id="locomotiveCarriage">

class locomotiveCarriage extends CGRAobject {
    constructor(glcontext, color1 = [1,0,0], color2 = [0, 0, 1], carriageSize = 2, mode=0){
        super(glcontext);
        
        this.carriageSize = carriageSize; // 1, 2, 3
        this.color1 = color1;
        this.color2 = color2;
        
        this.body = new cubeT(this.gl, color1, color2);
        this.lower_body = new cubeT(this.gl, color1, color2);

        this.roof = new cylinderT(this.gl, color1, color2);

        this.swheels = [];
        
        this.mode = mode;
        this.create_obj();
    }
    
    setShader(shaderprog){
        this.shader = shaderprog;
        
        this.body.setShader(shaderprog);
        this.lower_body.setShader(shaderprog);
        this.roof.setShader(shaderprog);
        
        for (let w of this.swheels) w.setShader(shaderprog);
    }
    
    setTexture(bodyTex, wheelTex, cgraTex = null) {
        this.bodyTex  = bodyTex;
        this.wheelTex = wheelTex;
        this.cgraTex = cgraTex;

        let useBodyTex = bodyTex;
        if (cgraTex) {
            useBodyTex = cgraTex;
        }
        this.body.setTexture(useBodyTex);
        
        this.lower_body.setTexture(bodyTex);
        this.roof.setTexture(bodyTex);

        for (let w of this.swheels) {
            w.setTexture(wheelTex);
        }
    }
    
    create_obj(){
        const lengthMap = {1: 3.0, 2: 6.0, 3: 9.0};
        const bodyLength = lengthMap[this.carriageSize] || 6.0;
        
        const c1 = this.color1;
        const c2 = this.color2;
        
        // lower_body
        let rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,1.0,0.0)));
        let scale = glm.scale(glm.mat4(1.0), glm.vec3(bodyLength, 1.0, 1.0));
        let pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 0.5, 0.0));
        let transform = pos['*'](rot)['*'](scale);
        this.lower_body.setModelTransformation(transform);

        // body
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,1.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(bodyLength, 2.5, 2.0));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.5, 0.0));
        transform = pos['*'](rot)['*'](scale);
        this.body.setModelTransformation(transform);

        
        this.swheels = [];
        let wheelZPositions;
        if(bodyLength === 3.0) wheelZPositions = [1.0, -1.0];
        else if(bodyLength === 6.0) wheelZPositions = [2.5, 1.25, -1.25, -2.5];
        else wheelZPositions = [3.75, 2.5, 1.25, -1.25, -2.5, -3.75];

        for(let z of wheelZPositions){
            for(let side of [-1,1]){ // left/right
                const color1 = side > 0 ? c1 : c2;
                const color2 = side > 0 ? c2 : c1;
                                    
                var wheel = new cylinderT(this.gl, color1, color2);
                                    
                if (this.mode==0){
                    wheel = new cylinderT(this.gl, color1, color2);
                } else if (this.mode==1){
                    wheel = new sphereT(this.gl, color1, color2);
                } else if (this.mode==2){
                    wheel = new cubeT(this.gl, color1, color2);
                }

                const rotW = glm.toMat4(glm.angleAxis(glm.radians(90.0), glm.vec3(0.0, 0.0, 1.0)));
                const scaleW = glm.scale(glm.mat4(1.0), glm.vec3(1.0, 0.5, 1.0));
                const posW = glm.translate(glm.mat4(1.0), glm.vec3(0.7*side, -0.25, z));
                wheel.baseTransform = posW['*'](rotW)['*'](scaleW);

                this.swheels.push(wheel);
            }
        }

        // roof
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(1.0,0.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(2.0, bodyLength, 2.0));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 2.5, 0.0));
        transform = pos['*'](rot)['*'](scale);
        this.roof.setModelTransformation(transform);

    }
    
    draw(viewM = glm.mat4(1.0), projectionM = glm.mat4(1.0), parentM = glm.mat4(1.0), counter = 0){
        
        this.lower_body.draw(viewM, projectionM, this.shader, parentM);
        this.body.draw(viewM, projectionM, this.shader, parentM);
        this.roof.draw(viewM, projectionM, this.shader, parentM);
        
        for (let w of this.swheels) {
            const angle = (counter % (2 * Math.PI)) * 2;
            const spin = glm.toMat4(glm.angleAxis(angle, glm.vec3(0.0,1.0,0.0)));
            w.setModelTransformation(w.baseTransform['*'](spin));
            w.draw(viewM, projectionM, this.shader, parentM);
        }
        
    }
}


</script>

In [ ]:
%%html
<script id="ellipsePose">

    function ellipsePose(a, b, alpha, t, eps) {
      // Parameter along ellipse
      const th = alpha * t + eps;

      // Position
      const x =  a * Math.cos(th);
      const z =  b * Math.sin(th);

      // Tangent
      let dx = -a * alpha * Math.sin(th);
      let dz =  b * alpha * Math.cos(th);

      const T = glm.translate(glm.mat4(1.0), glm.vec3(x, 0.75, z));
      const R = glm.toMat4(glm.angleAxis(Math.atan2(dx, dz), glm.vec3(0.0, 1.0, 0.0)));
      return T['*'](R);
    }

</script>

In [ ]:
%%html
<script id="train">

class train extends CGRAobject {
    constructor(glcontext, color1 = [1,0,0], color2 = [0, 0, 1], mode=0){
        super(glcontext);
        
        this.head = new locomotiveEngine(this.gl, color1, color2,mode);
        this.c1 = new locomotiveCarriage(this.gl, color1, color2, 2,mode);
        this.c2 = new locomotiveCarriage(this.gl, color1, color2, 1,mode);
        this.c3 = new locomotiveCarriage(this.gl, color1, color2, 3,mode);

        this.create_obj();
    }
    
    setShader(shaderprog){
        this.shader = shaderprog;
        
        this.head.setShader(shaderprog);
        this.c1.setShader(shaderprog);
        this.c2.setShader(shaderprog);
        this.c3.setShader(shaderprog);
    }
    
    setTexture(engineTex, bodyTex, wheelTex, cgraTex = null) {
        this.engineTex = engineTex;
        this.bodyTex  = bodyTex;
        this.wheelTex = wheelTex;
        this.cgraTex = cgraTex;

        this.head.setTexture(engineTex, wheelTex);
        this.c1.setTexture(bodyTex, wheelTex, cgraTex);
        this.c2.setTexture(bodyTex, wheelTex);
        this.c3.setTexture(bodyTex, wheelTex);
    }
    
    create_obj(){
       const I = glm.mat4(1.0);
       this.head.setModelTransformation(I);
       this.c1.setModelTransformation(I);
       this.c2.setModelTransformation(I);
       this.c3.setModelTransformation(I);
    }
    
    draw(viewM = glm.mat4(1.0), projectionM = glm.mat4(1.0), parentM = glm.mat4(1.0), M0 = glm.mat4(1.0), M1 = glm.mat4(1.0), M2 = glm.mat4(1.0), M3 = glm.mat4(1.0), counter = 0){
          const base = parentM['*'](this.modelMat);
          this.head.draw(viewM, projectionM, base['*'](M0), counter);
          this.c1.draw(viewM,  projectionM, base['*'](M1), counter);
          this.c2.draw(viewM,  projectionM, base['*'](M2), counter);
          this.c3.draw(viewM,  projectionM, base['*'](M3), counter);
    }
}


</script>

In [ ]:
%%html
<script id="palmTree">

class palmTree extends CGRAobject {
    constructor(glcontext, color1 = [1,0,0], color2 = [0, 0, 1]){
        super(glcontext);
        
        this.color1 = color1;
        this.color2 = color2;
        
        this.body1 = new cylinderT(this.gl, color1, color2);
        this.body2 = new sphereT(this.gl, color1, color2);
        
        this.leafs = [];
        
        this.create_obj();
    }
    
    setShader(shaderprog){       
        this.shader = shaderprog;
        
        this.body1.setShader(shaderprog);
        this.body2.setShader(shaderprog);
        
        for (let l of this.leafs) l.setShader(shaderprog);
    }
    
    setTexture(woodTex, leafTex) {
        this.woodTex  = woodTex;
        this.leafTex  = leafTex;

        this.body1.setTexture(woodTex);
        this.body2.setTexture(woodTex);
        
        for (let l of this.leafs) {
            l.setTexture(leafTex);
        }
    }
    
    create_obj(){
        
        const c1 = this.color1;
        const c2 = this.color2;
        
        // body1
        let rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,0.0,0.0)));
        let scale = glm.scale(glm.mat4(1.0), glm.vec3(1.0, 7.0, 1.0));
        let pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 3.5, 0.0));
        let transform = pos['*'](rot)['*'](scale);
        this.body1.setModelTransformation(transform);
        
        // body2
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0,0.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(1.5, 2.0, 1.5));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 7.0, 0.0));
        transform = pos['*'](rot)['*'](scale);
        this.body2.setModelTransformation(transform);
        
        // leafs
        const leafScales = [
            glm.vec3(5.0, 1.0, 1.0),
            glm.vec3(5.0, 1.0, 1.0),
            glm.vec3(5.0, 1.0, 1.0),
            glm.vec3(5.0, 1.0, 1.0),
            glm.vec3(5.0, 1.0, 1.0),
            glm.vec3(5.0, 1.0, 1.0),
            glm.vec3(5.0, 1.0, 1.0),
            glm.vec3(5.0, 1.0, 1.0)
        ];
        
        const leafAnglesX = [15, 15, 15, 15, 15, 15, 15, 15]; // degrees around X-axis
        const leafAnglesY = [0, 45, 90, 135, 180, 225, 270, 315]; // degrees around Y-axis

        this.leafs = [];

        for (let i = 0; i < leafScales.length; i++) {
            const c1 = [0.278,0.404,0.11];
            const c2 = [0.49,0.635,0.325];
            
            const leaf = new diskT(this.gl, c1, c2);

            const rotX = glm.toMat4(glm.angleAxis(glm.radians(90.0), glm.vec3(0.0, 1.0, 0.0)));
            const rotX2 = glm.toMat4(glm.angleAxis(glm.radians(leafAnglesX[i]), glm.vec3(1.0, 0.0, 0.0)));
            const rotY = glm.toMat4(glm.angleAxis(glm.radians(leafAnglesY[i]), glm.vec3(0.0, 1.0, 0.0)));

            const scale = glm.scale(glm.mat4(1.0), leafScales[i]);
            const pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 8.0, 0.0));
            const out = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 0.0, 1.75));

            const baseTransform = pos['*'](rotY)['*'](rotX2)['*'](out)['*'](rotX)['*'](scale);

            leaf.setModelTransformation(baseTransform);
            this.leafs.push(leaf);
        }
    }
    
    draw(viewM = glm.mat4(1.0), projectionM = glm.mat4(1.0), parentM = glm.mat4(1.0), counter = 0){
          const base = parentM['*'](this.modelMat);
        
          this.body1.draw(viewM, projectionM, this.shader, base);
          this.body2.draw(viewM, projectionM, this.shader, base);
        
          for (let l of this.leafs) l.draw(viewM, projectionM, this.shader, base);
     }

}


</script>

In [ ]:
%%html
<script id="obelisk">

class obelisk extends CGRAobject {
    constructor(glcontext, color1 = [1,0,0], color2 = [0, 0, 1]){
        super(glcontext);
        
        this.colum = new cubeT(this.gl, color1, color2);
        this.top = new pyramidT(this.gl, color1, color2);
        this.bottom = new pyramidT(this.gl, color1, color2);

        this.create_obj();
    }
    
    setShader(shaderprog){
        this.shader = shaderprog;
        
        this.colum.setShader(shaderprog);
        this.bottom.setShader(shaderprog);
        this.top.setShader(shaderprog);
    }
    
    setTexture(obeliskTex) {
        this.obeliskTex  = obeliskTex;

        this.colum.setTexture(obeliskTex);
        this.top.setTexture(obeliskTex);
        this.bottom.setTexture(obeliskTex);
    }
    
    create_obj(){
        
        var scale = glm.scale(glm.mat4(1.0), glm.vec3(1, 7, 1));
        var pos = glm.translate(glm.mat4(1.0), glm.vec3(0, 3.5, 0));
        var transform = pos['*'](scale);
        this.colum.setModelTransformation(transform);
        
        scale = glm.scale(glm.mat4(1.0), glm.vec3(1.3, 8, 1.3));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0, 4, 0));
        transform = pos['*'](scale);
        this.bottom.setModelTransformation(transform); 
        
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0, 7.5, 0));
        transform = pos;
        this.top.setModelTransformation(transform); 
    }
    
    draw(viewM = glm.mat4(1.0), projectionM = glm.mat4(1.0), parentM = glm.mat4(1.0)){
        
        this.colum.draw(viewM, projectionM, this.shader, parentM);
        this.top.draw(viewM, projectionM, this.shader, parentM);
        this.bottom.draw(viewM, projectionM, this.shader, parentM);
    }
}


</script>

In [ ]:
%%html
<script id="column">

class column extends CGRAobject {
    constructor(glcontext, color1 = [1,0,0], color2 = [0, 0, 1]){
        super(glcontext);
        
        this.colum = new cylinderT(this.gl, color1, color2);
        this.top = new coneT(this.gl, color2, color1);
        this.bottom = new coneT(this.gl, color1, color2);

        this.create_obj();
    }
    
    setShader(shaderprog){
        this.shader = shaderprog;
        
        this.colum.setShader(shaderprog);
        this.bottom.setShader(shaderprog);
        this.top.setShader(shaderprog);
    }
    
    setTexture(columnTex) {
        this.columnTex  = columnTex;

        this.colum.setTexture(columnTex);
        this.top.setTexture(columnTex);
        this.bottom.setTexture(columnTex);
    }
    
    create_obj(){
        
        var transform = glm.scale(glm.mat4(1.0), glm.vec3(1, 2, 1));
        this.colum.setModelTransformation(transform);
        
        var scale = glm.scale(glm.mat4(1.0), glm.vec3(1.5, 1, 1.5));
        var pos = glm.translate(glm.mat4(1.0), glm.vec3(0, -0.5, 0));
        transform = pos['*'](scale);
        this.bottom.setModelTransformation(transform); 
        
        var rot = glm.toMat4(glm.angleAxis(glm.radians(180.0),glm.vec3(1.0,0.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(1.5, 1, 1.5));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0, 0.5, 0));
        transform = pos['*'](rot)['*'](scale);
        this.top.setModelTransformation(transform); 
    }
    
    draw(viewM = glm.mat4(1.0), projectionM = glm.mat4(1.0), parentM = glm.mat4(1.0)){
        
        this.colum.draw(viewM, projectionM, this.shader, parentM);
        this.top.draw(viewM, projectionM, this.shader, parentM);
        this.bottom.draw(viewM, projectionM, this.shader, parentM);
    }
}


</script>

In [ ]:
%%html
<script id="camelNeck">

class camelNeck extends CGRAobject {
    constructor(glcontext, color1 = [1,0,0], color2 = [0, 0, 1]){
        super(glcontext);
        
        this.color1 = color1;
        this.color2 = color2;
        
        this.head1 = new cylinderT(this.gl, color1, color2);
        this.neck1 = new cylinderT(this.gl, color2, color1);
        this.neck2 = new cylinderT(this.gl, color2, color1);
        
        this.ears = [];
        
        this.create_obj();
    }
    
    setShader(shaderprog){
        this.shader = shaderprog;
        
        this.head1.setShader(shaderprog);
        this.neck1.setShader(shaderprog);
        this.neck2.setShader(shaderprog);
        
        for (let e of this.ears) e.setShader(shaderprog);
    }
    
    setTexture(camelTex) {
        this.camelTex = camelTex;

        this.head1.setTexture(camelTex);
        this.neck1.setTexture(camelTex);
        this.neck2.setTexture(camelTex);

        for (let e of this.ears) {
            e.setTexture(this.camelTex);
        }
    }
    
    create_obj(){
        const c1 = this.color1;
        const c2 = this.color2;
        
        // head1
        let rot = glm.toMat4(glm.angleAxis(glm.radians(90.0), glm.vec3(0.0,0.0,1.0)));
        let scale = glm.scale(glm.mat4(1.0), glm.vec3(0.75, 0.5, 1.25));
        let pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 2.2, 3.3));
        let transform = pos['*'](rot)['*'](scale);
        this.head1.setModelTransformation(transform);
        
        // neck1
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.9,0.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(0.75, 2.0, 1.0));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.25, 1.6));
        transform = pos['*'](rot)['*'](scale);
        this.neck1.setModelTransformation(transform);
        
        // neck2
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.6, 0.0, 0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(0.7, 1.25, 1.0));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.7, 2.7));
        transform = pos['*'](rot)['*'](scale);
        this.neck2.setModelTransformation(transform);
        
        // ears
        const earPositions = [
            glm.vec3(-0.3, 2.4, 3.0),   // right
            glm.vec3(0.3, 2.4, 3.0)   // left
        ];
        
        const earRotations = [
            glm.radians(90.0),
            glm.radians(270.0)
        ];

        this.ears = [];

        for (let i = 0; i < earPositions.length; i++) {
        
            const ear = new cylinderT(this.gl, c1, c2);
            
            const rot = glm.toMat4(glm.angleAxis(earRotations[i], glm.vec3(0.0, 0.0, 0.75)));
            const scale = glm.scale(glm.mat4(1.0), glm.vec3(0.2, 0.4, 0.2));
            const pos = glm.translate(glm.mat4(1.0), earPositions[i]);
            const baseTransform = pos['*'](rot)['*'](scale);
            
            ear.setModelTransformation(baseTransform);
            this.ears.push(ear);
        }
        
    }
    
    draw(viewM = glm.mat4(1.0), projectionM = glm.mat4(1.0), parentM = glm.mat4(1.0), counter = 0){
          const base = parentM['*'](this.modelMat);
          
          this.head1.draw(viewM, projectionM, this.shader, base);

          this.neck1.draw(viewM, projectionM, this.shader, base);
          this.neck2.draw(viewM, projectionM, this.shader, base);
        
          for (let e of this.ears) e.draw(viewM, projectionM, this.shader, base);
     }

}


</script>

In [ ]:
%%html
<script id="camel">

class camel extends CGRAobject {
    constructor(glcontext, color1 = [1,0,0], color2 = [0, 0, 1]){
        super(glcontext);
        
        this.color1 = color1;
        this.color2 = color2;
        
        this.body1 = new cylinderT(this.gl, color2, color1);
        this.tail = new cylinderT(this.gl, color2, color1);
        
        this.butt = new sphereT(this.gl, color1, color2);
        
        this.legs = [];
        this.humps = [];
        
        this.neck = new camelNeck(this.gl, color1, color2);
        
        this.create_obj();
    }
    
    setShader(shaderprog){
        this.shader = shaderprog;
        
        this.body1.setShader(shaderprog);
        this.tail.setShader(shaderprog);
        
        this.butt.setShader(shaderprog);
        
        for (let l of this.legs) l.setShader(shaderprog);
        for (let h of this.humps) h.setShader(shaderprog);
        
        this.neck.setShader(shaderprog);
    }
    
    setTexture(camelTex) {
        this.camelTex = camelTex;

        this.body1.setTexture(camelTex);
        this.tail.setTexture(camelTex);
        this.butt.setTexture(camelTex);

        for (let l of this.legs)  l.setTexture(camelTex);
        for (let h of this.humps) h.setTexture(camelTex);

        this.neck.setTexture(camelTex);
    }
    
    create_obj(){
        const c1 = this.color1;
        const c2 = this.color2;
        
        // body1
        let rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(1.0,0.0,0.0)));
        let scale = glm.scale(glm.mat4(1.0), glm.vec3(1.8, 3.5, 2.3));
        let pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.25, 0.0));
        let transform = pos['*'](rot)['*'](scale);
        this.body1.setModelTransformation(transform);
        
        // tail
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.4,0.0,0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(0.25, 1.5, 0.25));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.3, -2.3));
        transform = pos['*'](rot)['*'](scale);
        this.tail.setModelTransformation(transform);
        
        // butt
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0, 0.0, 0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(1.6, 2.4, 2.4));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 1.25, -1.25));
        transform = pos['*'](rot)['*'](scale);
        this.butt.setModelTransformation(transform);
        
        // legs
        const legPositions = [
            glm.vec3(0.6, 0.0, 1.5),   
            glm.vec3(0.6, 0.0, -1.5),
            glm.vec3(-0.6, 0.0, 1.5),
            glm.vec3(-0.6, 0.0, -1.5)
        ];

        this.legs = [];

        for (let i = 0; i < legPositions.length; i++) {
        
            const leg = new cylinderT(this.gl, c1, c2);
            
            const rot = glm.toMat4(glm.angleAxis(glm.radians(90.0), glm.vec3(0.0, 0.0, 0.0)));
            const scale = glm.scale(glm.mat4(1.0), glm.vec3(0.5, 3.0, 0.5));
            const pos = glm.translate(glm.mat4(1.0), legPositions[i]);
            const baseTransform = pos['*'](rot)['*'](scale);
            
            leg.setModelTransformation(baseTransform);
            this.legs.push(leg);
        }
        
        // humps
        const humpPositions = [
            glm.vec3(0.0, 1.9, -1.0),
            glm.vec3(0.0, 2.0, 0.75)
        ];
        
        const humpScales = [
            glm.vec3(2.5, 1.2, 1.5),
            glm.vec3(2.5, 1.2, 2)
        ];

        this.humps = [];

        for (let i = 0; i < humpPositions.length; i++) {
        
            const hump = new sphereT(this.gl, c2, c1);
            
            const rot = glm.toMat4(glm.angleAxis(glm.radians(270.0), glm.vec3(0.0, 0.0, 1.0)));
            const scale = glm.scale(glm.mat4(1.0), humpScales[i]);
            const pos = glm.translate(glm.mat4(1.0), humpPositions[i]);
            const baseTransform = pos['*'](rot)['*'](scale);
            
            hump.setModelTransformation(baseTransform);
            this.humps.push(hump);
        }
        
        // neck
        rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(0.0, 0.0, 0.0)));
        scale = glm.scale(glm.mat4(1.0), glm.vec3(1.0, 1.0, 1.0));
        pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 0.0, 0.0));
        transform = pos['*'](rot)['*'](scale);
        this.neck.setModelTransformation(transform);
              
    }
    
    setNeckAngle(angle){
      let rot = glm.toMat4(glm.angleAxis(glm.radians(90.0),glm.vec3(angle, 0.0, 0.0)));
      let scale = glm.scale(glm.mat4(1.0), glm.vec3(1.0, 1.0, 1.0));
      let pos = glm.translate(glm.mat4(1.0), glm.vec3(0.0, 0.0, 0.0));
      let transform = pos['*'](rot)['*'](scale);
      this.neck.setModelTransformation(transform);
    }
    
    draw(viewM = glm.mat4(1.0), projectionM = glm.mat4(1.0), parentM = glm.mat4(1.0), counter = 0){
          const base = parentM;
          
          const t = counter/5; //~30 second loop  (5 times slower than a normal loop, 6.28s)
          const angle = 0.3 * (Math.sin(t) + 1.0);  // normalize between [0.3, 0.6] to limit neck position

          this.setNeckAngle(angle);
        
          this.body1.draw(viewM, projectionM, this.shader, base);

          this.tail.draw(viewM, projectionM, this.shader, base);
        
          this.butt.draw(viewM, projectionM, this.shader, base);

          for (let l of this.legs) l.draw(viewM, projectionM, this.shader, base);
          for (let h of this.humps) h.draw(viewM, projectionM, this.shader, base);
        
          this.neck.draw(viewM, projectionM, base, counter = 0);
     }

}


</script>

### Main

Deteta a os marcadores Aruco e desenha os objetos correspondentes. ID 0 = Cubo, ID 1 = Esfera, ID 2 = Camelo e oasis, ID 3 = Comboio e pista, ID 4 = Boneco em cima da carruagem

In [ ]:
%%html

<pre id=testfield> </pre>
<pre id=markerfield></pre>

<script>
class myapp extends DEECapp{
    counter=0;
    
    initvideofile(){
        this.video = document.getElementById("videofile");
        this.back = document.createElement('canvas');
        this.backctx = this.back.getContext('2d');
    }
    initcamera(){
        this.video = document.getElementById("video");
        this.back = document.createElement('canvas');
        this.backctx = this.back.getContext('2d');
      if (navigator.mediaDevices === undefined) {
        navigator.mediaDevices = {};
      }
      
      if (navigator.mediaDevices.getUserMedia === undefined) {
        navigator.mediaDevices.getUserMedia = function(constraints) {
          var getUserMedia = navigator.webkitGetUserMedia || navigator.mozGetUserMedia;
          
          if (!getUserMedia) {
            return Promise.reject(new Error('getUserMedia is not implemented in this browser'));
          }

          return new Promise(function(resolve, reject) {
            getUserMedia.call(navigator, constraints, resolve, reject);
          });
        }
      }
      
      navigator.mediaDevices
        .getUserMedia({ video: true })
        .then(function(stream) {
          if ("srcObject" in video) {
            video.srcObject = stream;
          } else {
            video.src = window.URL.createObjectURL(stream);
          }
        })
        .catch(function(err) {
          console.log(err.name + ": " + err.message);
        }
      );
    }
    
    initialize(){

        if ( document.getElementById("videofile")){
            this.initvideofile();
        } 
        else {
            this.initcamera();
        }

        var fragsrcT = document.getElementById("my-fragment-shaderT").text;
        var vertsrcT = document.getElementById("my-vertex-shaderT").text;
        
        this.shaderprogT = new DEECshader(this.gl);
        this.shaderprogT.srcShaders(vertsrcT,fragsrcT);
        
        var frag_bones = document.getElementById("bones-fragment-shader").text;
        var vert_bones = document.getElementById("bones-vertex-shader").text;
        this.shaderBones = new DEECshader(this.gl);
        this.shaderBones.srcShaders(vert_bones, frag_bones);
      
        
        // perform other initializations
        this.gl.enable(this.gl.DEPTH_TEST);
        this.gl.clearColor(0.30,0.30,0.30,1.0);
        
        // Projection and view matrices
        
        let fy = 824.3715784522805;
        let height = 720.0;

        let fovy = 2.0 * Math.atan(height / (2.0 * fy));
        let aspect = 1280.0 / 720.0;
        let near = 0.1;
        let far  = 1000.0;

        this.projM = glm.perspective(fovy, aspect, near, far);


        this.viewM = glm.lookAt(glm.vec3(0,0,0),glm.vec3(0,0,-1),glm.vec3(0,1,0));

        //=======================================================
        // Initialize objects
        
        this.markerObjects = [];  
        this.modelAR = [];
        
        this.BackgroundVideo = new BackgroundT(this.gl);
        this.BackgroundVideo.setShader(this.shaderprogT);
        
        
        this.markerObjects[0] = new cubeT(this.gl, [1,0,0], [0,0,1]);
        this.markerObjects[0].setShader(this.shaderprogT);
        
        this.markerObjects[1] = new sphereT(this.gl, [1,0,0], [0,0,1]);
        this.markerObjects[1].setShader(this.shaderprogT);
        
        this.texture1 = new CGRAtexture(this.gl);
        this.texture1.load("Textures/train_metal.jpg");

        this.markerObjects[0].setTexture(this.texture1);
        
        this.texture2 = new CGRAtexture(this.gl);
        this.texture2.load("Textures/water.jpg");

        this.markerObjects[1].setTexture(this.texture2);
        
        this.markerObjects[2] = new camel(this.gl, [0.439, 0.314, 0.184], [0.757, 0.604, 0.420]);
        this.markerObjects[2].setShader(this.shaderprogT);
        
        this.camelTexture = new CGRAtexture(this.gl);
        this.camelTexture.load("Textures/camel1.jpg");
        this.camelTexture2 = new CGRAtexture(this.gl);
        this.camelTexture2.load("Textures/camel5.jpg");
        this.markerObjects[2].setTexture(this.camelTexture);
        
        //Train
        this.markerObjects[3] = new train(this.gl, [0, 0, 0], [0.450, 0.00450, 0.0193]);
        this.markerObjects[3].setShader(this.shaderprogT);
        
        this.engineTexture = new CGRAtexture(this.gl);
        this.engineTexture.load("Textures/locomotive_metal2.jpg");
        
        this.bodyTexture = new CGRAtexture(this.gl);
        this.bodyTexture.load("Textures/train_metal.jpg");

        this.cgraTexture = new CGRAtexture(this.gl);
        this.cgraTexture.load("Textures/cgra_texture2.jpg");
        
        this.wheelTexture = new CGRAtexture(this.gl);
        this.wheelTexture.load("Textures/locomotive_metal2.jpg");
        
        this.markerObjects[3].setTexture(this.engineTexture, this.bodyTexture, this.wheelTexture, this.cgraTexture);
        
        
        //Lakes
        this.lake = new diskT(this.gl, [0.460, 0.866, 0.920],[1,1,1]);
        this.lake.setShader(this.shaderprogT);
        
        this.oasis = new diskT(this.gl, [0.356, 0.690, 0.235], [0.356, 0.690, 0.235]);
        this.oasis.setShader(this.shaderprogT);
        
        this.waterTexture = new CGRAtexture(this.gl);
        this.waterTexture.load("Textures/water.jpg");
        this.grassTexture = new CGRAtexture(this.gl);
        this.grassTexture.load("Textures/grass.jpg");
        this.lake.setTexture(this.waterTexture);
        this.oasis.setTexture(this.grassTexture);
        
         //Floor and track
        this.floor = new diskT(this.gl, [0.886, 0.792, 0.463], [0.886, 0.792, 0.463]);
        this.floor.setShader(this.shaderprogT);
        
        this.inner_floor = new diskT(this.gl, [0.886, 0.792, 0.463], [0.886, 0.792, 0.463], 60);
        this.inner_floor.setShader(this.shaderprogT);
        
        this.track = new diskT(this.gl, [0.6,0.6,0.6], [0.6,0.6,0.6], 60);
        this.track.setShader(this.shaderprogT);
        
        this.groundTexture = new CGRAtexture(this.gl);
        this.groundTexture.load("Textures/sand8.jpg");
        this.floor.setTexture(this.groundTexture);
        this.inner_floor.setTexture(this.groundTexture);
        
        this.trackTexture = new CGRAtexture(this.gl);
        this.trackTexture.load("Textures/gravel2.jpg");
        this.track.setTexture(this.trackTexture);
        
        
        //Camels
        this.camel = new camel(this.gl, [0.439, 0.314, 0.184], [0.757, 0.604, 0.420]);
        this.camel.setShader(this.shaderprogT);
        
        this.camel2 = new camel(this.gl, [0.439, 0.314, 0.184], [0.757, 0.604, 0.420]);
        this.camel2.setShader(this.shaderprogT);
        
        this.camel3 = new camel(this.gl, [0.439, 0.314, 0.184], [0.757, 0.604, 0.420]);
        this.camel3.setShader(this.shaderprogT);
        
        this.camelTexture = new CGRAtexture(this.gl);
        this.camelTexture.load("Textures/camel1.jpg");
        this.camel.setTexture(this.camelTexture);
        this.camel2.setTexture(this.camelTexture);
        this.camel3.setTexture(this.camelTexture);
        
        
        //Palmtree
        this.palmtrees = [];
        this.palmtrees_trans = [];
        
        this.trunkTexture = new CGRAtexture(this.gl);
        this.trunkTexture.load("Textures/tree_trunk.jpg");
        this.leafTexture = new CGRAtexture(this.gl);
        this.leafTexture.load("Textures/tree_leaf2.jpg");
        for (let i = 0; i < 2; i++) {
            this.palmtrees[i] = new palmTree(this.gl, [0.42,0.286,0.169], [0.714,0.592,0.49]);
            this.palmtrees[i].setShader(this.shaderprogT);
            
            this.palmtrees[i].setTexture(this.trunkTexture, this.leafTexture);
        }
        
        //Obelisk
        this.obelisk = new obelisk(this.gl, [0.800,0.570,0.0320], [0.950,0.816,0.333]);
        this.obelisk.setShader(this.shaderprogT);
        
        this.columnTexture = new CGRAtexture(this.gl);
        this.columnTexture.load("Textures/sandstone.jpg");        
        this.obelisk.setTexture(this.columnTexture);
        
        //Human
        this.human = new human(this.gl, [0.8, 0.1, 0.1], [0.1, 0.1, 0.8]);
        this.human.setShaders(this.shaderprogT, this.shaderBones);
        //=================================================================
        this.trans_track = glm.translate(glm.mat4(1.0),glm.vec3(0,0.01,0));
        this.trans_inner_floor = glm.translate(glm.mat4(1.0),glm.vec3(0,0.02,0));
        
        this.floor_scale = glm.scale(glm.mat4(1.0),glm.vec3(25,0,25));
        this.inner_floor_scale = glm.scale(glm.mat4(1.0),glm.vec3(24-8,0,18-8));
        
        this.track_scale = glm.scale(glm.mat4(1.0),glm.vec3(24,0,18));
        
        this.half_scale = glm.scale(glm.mat4(1.0),glm.vec3(0.5,0.5,0.5));
        this.lake_scale = glm.scale(glm.mat4(1.0),glm.vec3(9,0.1,7));
        
        this.oasis_scale = glm.scale(glm.mat4(1.0),glm.vec3(12.5,0.1,15));
        
        
        //Camel
        this.trans_camel = glm.translate(glm.mat4(1.0),glm.vec3(0,1.2,0));
        this.trans_camel2 = glm.translate(glm.mat4(1.0),glm.vec3(3,0.6,2));
        this.trans_camel3 = glm.translate(glm.mat4(1.0),glm.vec3(-10,1.2,8));
        
        this.trans_lake = glm.translate(glm.mat4(1.0),glm.vec3(0,0.05,6));
        this.trans_oasis = glm.translate(glm.mat4(1.0),glm.vec3(0,0.04,6));
        
        this.palmtrees_trans[0] = glm.translate(glm.mat4(1.0),glm.vec3(0,0,15));
        this.palmtrees_trans[1] = glm.translate(glm.mat4(1.0),glm.vec3(-5,0,0));
    
        this.trans_obelisk = glm.translate(glm.mat4(1.0),glm.vec3(3,0,11));
        
        //===================================================================
        this.texture3 = new CGRAtexture(this.gl);
        
        this.BackgroundVideo.settexture(this.texture3);
    
        // HERE IS the initalization of the Aruco detector
        this.detector = new AR.Detector();
        var modelsize = 35.0;
        var focallength = 680.47103923802;
        // and of the pose estimator
        this.posit = new POS.Posit(modelsize,focallength);
    }
    
    calculateSize(srcSize, dstSize) {
    var srcRatio = srcSize.width / srcSize.height;
    var dstRatio = dstSize.width / dstSize.height;
    if (dstRatio > srcRatio) {
      return {
        width:  dstSize.height * srcRatio,
        height: dstSize.height
      };
    } else {
      return {
        width:  dstSize.width,
        height: dstSize.width / srcRatio
      };
    }
  }
    
    processvideo(){
        
        if (this.video.readyState === this.video.HAVE_ENOUGH_DATA) {
            var videoSize = { width: this.video.videoWidth, height: this.video.videoHeight };
            var canvasSize = { width: this.canvas.width, height: this.canvas.height };
            var renderSize = this.calculateSize(videoSize, canvasSize);
            var xOffset = (canvasSize.width - renderSize.width) / 2;
          
            this.back.width=this.video.videoWidth;
            this.back.height=this.video.videoHeight;
            this.backctx.drawImage(this.video,0,0,this.video.videoWidth,this.video.videoHeight);
            var imagedata = this.backctx.getImageData(0,0,this.video.videoWidth,this.video.videoHeight);
            
            this.texture3.update(imagedata,this.video.videoWidth,this.video.videoHeight);    
            //
            // Detect the markers 
            var markers = this.detector.detect(imagedata);
            
            this.modelAR = []; // Reset markers every iteration
          var mText ="Markers detected = " + markers.length + "\n";
            if (markers.length > 0){
                for (var j = 0; j !== markers.length; ++ j){
                    this.id=markers[j].id;
                    
                    mText += "Marker "+j + " ID= " + markers[j].id +"\n";
                    var corners = markers[j].corners;
                
                    for (var i=0; i !== corners.length; ++ i){
                        var corner = corners[(i + 1) % corners.length];
                        mText += "Corner " + i + " " + corner.x + " " + corner.y + "\n";
                    }
	
                
                var pose = this.posit.pose(corners);
                var translation = pose.bestTranslation;
                var rotation = pose.bestRotation;
                    

               var txtText = "Rotation = \n";
                for (var l=0;l<3;l++){
                    testfield.innerText += "|";
                    for (var c=0;c<3;c++){

                        txtText += Math.round(rotation[l][c]*100)/100;
                        txtText += ", ";
                    }
                    txtText += "|\n";
                }
        
                
                // pose.bestRotation, pose.bestTranslation);
                txtText += "Translation = \n [" + Math.round(translation[0]*100)/100 + ", "+ Math.round(translation[1]*100)/100 +", "+ Math.round(translation[2]*100)/100 + " ]";
                testfield.innerText = txtText;
                markerfield.innerText=mText;
                    

                // Convert rotation to glm.mat4
                let R = glm.mat4(
                    rotation[0][0], rotation[0][1], rotation[0][2], 0,
                    rotation[1][0], rotation[1][1], rotation[1][2], 0,
                    rotation[2][0], rotation[2][1], rotation[2][2], 0,
                    0,              0,              0,              1
                );

                // Convert translation to OpenGL coordinates
                let T = glm.translate(
                    glm.mat4(1.0),
                    glm.vec3(
                        (translation[0])/340-0.5,    
                       -(translation[1])/203+0.5,     
                       -translation[2]/700      
                    )
                );

                // Compose final model matrix for the corresponding object
                this.modelAR[this.id] = T['*'](R['*'](glm.scale(glm.mat4(1.0), glm.vec3(1/35, -1/35, 1/35))));
                }
            }
        }      
    }

    
    render(){
        this.gl.clear(this.gl.COLOR_BUFFER_BIT | this.gl.DEPTH_BUFFER_BIT);
        this.processvideo();
        this.counter++;
        
        // Draw Background
        this.BackgroundVideo.drawit();
        
        
        
        //Draw objects if there are models according to the markers detected
        //The model calculated according to the marker position is ccombined with the transforms previously applied in the last assignment
        
        for (this.id in this.modelAR){
            if(this.id==0 || this.id==1){ //Cube and sphere
                this.markerObjects[this.id].setModelTransformation(this.modelAR[this.id]);
                this.markerObjects[this.id].draw(this.viewM, this.projM, this.shaderprogT);
            }
            if(this.id==2){ //Camel and rest of oasis
                this.markerObjects[this.id].draw(this.viewM, this.projM, this.modelAR[this.id]['*'](this.trans_camel),this.counter/3);
                
                //lakes   
                this.lake.setModelTransformation(this.modelAR[this.id]['*'](this.trans_lake['*'](this.lake_scale)));
                this.lake.draw(this.viewM, this.projM, this.shaderprogT);
                this.oasis.setModelTransformation(this.modelAR[this.id]['*'](this.trans_oasis['*'](this.oasis_scale)));
                this.oasis.draw(this.viewM, this.projM, this.shaderprogT);
                this.camel2.draw(this.viewM, this.projM, this.modelAR[this.id]['*'](this.trans_camel2['*'](this.half_scale)), 8);
                this.camel3.draw(this.viewM, this.projM, this.modelAR[this.id]['*'](this.trans_camel3['*'](glm.toMat4(glm.angleAxis(glm.radians(135), glm.vec3(0.0, 1.0, 0.0))))),22);

                for (let i = 0; i < this.palmtrees.length; i++) {
                    this.palmtrees[i].draw(this.viewM, this.projM, this.modelAR[this.id]['*'](this.palmtrees_trans[i]));
                }
                
                //obelisk
                this.obelisk.draw(this.viewM, this.projM, this.modelAR[this.id]['*'](this.trans_obelisk));
        
            }
            if(this.id==3){ //Train and track
                //Set size of track an speed
                this.a=10;
                this.b=7;
                this.alpha=1/20;

                const I = glm.mat4(1.0);

                //train starts at -pi/2
                const M0 = ellipsePose(this.a, this.b, this.alpha, this.counter, -Math.PI/2);
                const M1 = ellipsePose(this.a, this.b, this.alpha, this.counter,  -3*Math.PI/4);
                const M2 = ellipsePose(this.a, this.b, this.alpha, this.counter,  -4*Math.PI/4);
                const M3 = ellipsePose(this.a, this.b, this.alpha, this.counter,  -5*Math.PI/4);

                this.markerObjects[this.id].draw(this.viewM, this.projM, this.modelAR[this.id], M0, M1, M2, M3, -this.counter);
                
                //Track and floor
                this.floor.setModelTransformation(this.modelAR[this.id]['*'](this.floor_scale));
                this.floor.draw(this.viewM, this.projM, this.shaderprogT);

                this.inner_floor.setModelTransformation(this.modelAR[this.id]['*'](this.trans_inner_floor['*'](this.inner_floor_scale)));
                this.inner_floor.draw(this.viewM, this.projM, this.shaderprogT);
        
                //Track
                this.track.setModelTransformation(this.modelAR[this.id]['*'](this.trans_track['*'](this.track_scale)));
                this.track.draw(this.viewM, this.projM, this.shaderprogT);
                
            }
            if(this.id==4){ //Dancing humanoid
                //Set size of track an speed
                this.a=10;
                this.b=7;
                this.alpha=1/20;

                const I = glm.mat4(1.0);

                //train starts at -pi/2
                const M0 = ellipsePose(this.a, this.b, this.alpha, 0, -Math.PI/2);
                const M1 = ellipsePose(this.a, this.b, this.alpha, 0,  -3*Math.PI/4);
                const M2 = ellipsePose(this.a, this.b, this.alpha, 0,  -4*Math.PI/4);
                const M3 = ellipsePose(this.a, this.b, this.alpha, 0,  -5*Math.PI/4);

                this.markerObjects[this.id-1].draw(this.viewM, this.projM, this.modelAR[this.id], M0, M1, M2, M3, 0);
                
                // human
                let humanPos = M1['*'](glm.translate(glm.mat4(1.0), glm.vec3(0.0, 3.5, 0.0)));
                humanPos = humanPos['*'](glm.scale(glm.mat4(1.0), glm.vec3(0.75, 0.75, 0.75)));

                this.human.setModelTransformation(this.modelAR[this.id]['*'](humanPos));

                this.human.draw(this.viewM, this.projM, glm.mat4(1.0), -this.counter/3);
                
            }
        }

    }
}

var app = new myapp('myCanvas');

app.run();

</script>